In [1]:
!pip install -q "/kaggle/input/pythonlibrarieswheelfiles/lightgbm-4.2.0-py3-none-manylinux_2_28_x86_64.whl"

In [2]:
# %load_ext cudf.pandas
import pandas as pd
# import cudf as pd

from sklearn.model_selection import TimeSeriesSplit, GroupKFold
from sklearn.metrics import mean_absolute_error as MAE
# from sklearn.metrics import mean_absolute_error

# from xgboost import XGBRegressor
# import catboost as cb
import lightgbm as lgb

import datetime
import time
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt

In [3]:
print(lgb.__version__)

4.2.0


In [4]:
import platform

if platform.release() == '5.15.133.1-microsoft-standard-WSL2':
    comp_path = "./predict-energy-behavior-of-prosumers"
else:
    comp_path = "/kaggle/input/predict-energy-behavior-of-prosumers"


import json
with open(comp_path + "/county_id_to_name_map.json") as file:
    county_codes = json.load(file)

In [5]:
import numpy as np
train = pd.read_csv(comp_path + "/train.csv")
elec_prices = pd.read_csv(comp_path + "/electricity_prices.csv")
# gas_prices_train = pd.read_csv(comp_path +  "/gas_prices.csv")
client_train = pd.read_csv(comp_path + "/client.csv")

if platform.release() == '5.15.133.1-microsoft-standard-WSL2':
    county_coordinate_weight_path = "./enefit-county-coordinate-weights"
else:
    county_coordinate_weight_path = "/kaggle/input/enefit-county-coordinate-weights"



county_coordinate_weights = pd.read_csv(county_coordinate_weight_path + "/county_pixel_weights.csv")

train.datetime = pd.to_datetime(train.datetime)
train['date'] = pd.to_datetime(train.datetime).dt.date
# train['date'] = pd.to_datetime(train.datetime).dt.floor("D") # workaround for cudf


# elec_prices.forecast_date = pd.to_datetime(elec_prices.forecast_date).dt.date
# gas_prices.forecast_date = pd.to_datetime(gas_prices.forecast_date).dt.date
train.head(2)

,county,is_business,product_type,target,is_consumption,datetime,data_block_id,row_id,prediction_unit_id,date
0,0,0,1,0.713,0,2021-09-01,0,0,0,2021-09-01
1,0,0,1,96.590,1,2021-09-01,0,1,0,2021-09-01


In [6]:
client_train['date'] = pd.to_datetime(client_train.date).dt.date
# client_train['date'] = pd.to_datetime(client_train.date).dt.floor("D") # workaround for cudf

client_train['available_date'] = client_train['date'] + datetime.timedelta(days=2)

client_features = ['eic_count', 'installed_capacity']

## Weather features

In [7]:
from functools import reduce
from itertools import chain

def get_weighted_historical_weather_by_county(forecast_weather_df, time_column='datetime', shift_feature=True):

    weather_condition_columns = [
        'temperature',
        'dewpoint',
        'cloudcover_high',
        'cloudcover_low',
        'cloudcover_mid',
        'cloudcover_total',
        'direct_solar_radiation',
        'diffuse_radiation',
        'shortwave_radiation',
        'snowfall', 
        'rain',
    ]
    
    cumulative_features = [
        'direct_solar_radiation',
        'diffuse_radiation',
        'snowfall', 
        'rain',
    ]
    
    point_features = [
        'temperature',
        'dewpoint',
        'cloudcover_high',
        'cloudcover_low',
        'cloudcover_mid',
        'cloudcover_total',
    ]
    
    county_coordinate_weights_truncated = county_coordinate_weights.copy()
    
    new_df = forecast_weather_df.copy()
    
    max_time = new_df['datetime'].max()
    
    
    new_df[['latitude', 'longitude']] = new_df[['latitude', 'longitude']].astype(np.float64).round(1)
    
    # The forecast data served during the test phase may have duplicate/missing coordinate pairs
    # Need to reindex the dataframe to avoid errors
    coors = historical_weather_train[['latitude', 'longitude']].set_index(['latitude', 'longitude']).sort_index().index.unique()
    timestamps = new_df[time_column].unique()
    idx = [[ts] + [*coor] for ts in timestamps for coor in coors]
    reidx = pd.DataFrame(idx, columns=['datetime', 'latitude', 'longitude']).set_index(['datetime', 'latitude', 'longitude']).index
    
    weather_condition_arr = new_df.sort_values(
                            [time_column, 'latitude', 'longitude']
                            ).groupby(
                                [time_column, 'latitude', 'longitude']
                                )[weather_condition_columns].mean()
    
    weather_condition_arr = weather_condition_arr.reindex(reidx).values.reshape(-1, 112, len(weather_condition_columns))
    
    
    new_index = new_df.sort_values([time_column]).groupby(
                            [time_column]
                            )[weather_condition_columns].mean().index.drop_duplicates()


    new_index = np.repeat(new_index, 16)

    
    weather_values_arr = (np.swapaxes(weather_condition_arr, 1, 2)  @ county_coordinate_weights_truncated.set_index('county_id').values.T)
    weather_values_arr = np.swapaxes(weather_values_arr, 1, 2).reshape(-1, len(weather_condition_columns))
    
    
    weather_by_county = pd.DataFrame(weather_values_arr, columns=weather_condition_columns, index=new_index)
    weather_by_county['county'] = np.array(np.arange(16).tolist() * (weather_values_arr.shape[0]//16) )
    
    weather_by_county.reset_index(inplace=True)
    
    
    weather_by_county = weather_by_county.set_index(['county', 'datetime'])#.sort_index()
    
    weather_by_county = weather_by_county.reindex(
                            weather_by_county.index.tolist() + [(county, max_time + datetime.timedelta(hours=i)) for county in range(16) for i in range(1, 49)]
                        )
    
    weather_by_county = weather_by_county.sort_index()

    if shift_feature:
#         weather_by_county[cumulative_features] = weather_by_county[cumulative_features].shift(-1)
        weather_by_county[point_features] = (weather_by_county[point_features].shift(1) + weather_by_county[point_features]) / 2
        
    
    lag_weather_features = [
        'direct_solar_radiation', 
        'cloudcover_total',
        'diffuse_radiation',
        # 'shortwave_radiation',
#         'temperature',
#         'cloudcover_low',
#         'dewpoint',
        ]
    
    for lag_day in [2, 3, 7, 14]:
        weather_by_county[[f"{ft}_lag_{lag_day}_day" for ft in lag_weather_features]] = weather_by_county[lag_weather_features].shift(24*lag_day)
        weather_by_county[[f"{ft}_{lag_day}_day_diff" for ft in lag_weather_features]] = weather_by_county[lag_weather_features].values  - weather_by_county[[f"{ft}_lag_{lag_day}_day" for ft in lag_weather_features]].values 


    weather_by_county[[f"{ft}_7_day_mean" for ft in lag_weather_features]] = sum([weather_by_county[lag_weather_features].shift(24*i) for i in range(2, 9)]) / 7
    weather_by_county[[f"{ft}_7_day_1h_mean" for ft in lag_weather_features]] = sum([weather_by_county[lag_weather_features].shift(24*i+1) for i in range(2, 9)]) / 7
    
    weather_by_county['temp_minus_dewpoint'] = weather_by_county['temperature'] - weather_by_county['dewpoint']
    weather_by_county['cloudcover_total'] = weather_by_county['cloudcover_total'] / 100
    
    weather_by_county = weather_by_county.reset_index()
    
    weather_by_county = weather_by_county.rename(columns={'temperature': 'temperature_h',
                                                         'dewpoint': 'dewpoint_h',
                                                         'direct_solar_radiation': 'direct_solar_radiation_h',
                                                          'direct_solar_radiation_lag_2_day': 'direct_solar_radiation_lag_2_day_h',
                                                          'direct_solar_radiation_lag_3_day': 'direct_solar_radiation_lag_3_day_h',
                                                          'direct_solar_radiation_lag_7_day': 'direct_solar_radiation_lag_7_day_h',
                                                          'direct_solar_radiation_7_day_mean': 'direct_solar_radiation_7_day_mean_h',
                                                          'direct_solar_radiation_7_day_1h_mean': 'direct_solar_radiation_7_day_1h_mean_h',
                                                          'temp_minus_dewpoint': 'temp_minus_dewpoint_h',
                                                          'cloudcover_total_lag_2_day': 'cloudcover_total_lag_2_day_h',
                                                          'cloudcover_total_lag_3_day': 'cloudcover_total_lag_3_day_h',
                                                          'cloudcover_total_lag_7_day': 'cloudcover_total_lag_7_day_h',
                                                          'cloudcover_total_7_day_mean': 'cloudcover_total_7_day_mean_h',
                                                          'diffuse_radiation_lag_2_day': 'diffuse_radiation_lag_2_day_h',
                                                          'diffuse_radiation_lag_3_day': 'diffuse_radiation_lag_3_day_h',
                                                          'diffuse_radiation_lag_7_day': 'diffuse_radiation_lag_7_day_h',
                                                          'diffuse_radiation_7_day_1h_mean': 'diffuse_radiation_7_day_1h_mean_h',
                                                          # 'shortwave_radiation_lag_2_day': 'shortwave_radiation_lag_2_day_h',
                                                          # 'shortwave_radiation_lag_7_day': 'shortwave_radiation_lag_7_day_h',
                                                         })
    return weather_by_county[['county', 
                              'datetime', 
                              'direct_solar_radiation_lag_2_day_h', 
                              'direct_solar_radiation_lag_3_day_h', 
                              'direct_solar_radiation_lag_7_day_h', 
                              'direct_solar_radiation_7_day_mean_h', 
                              'direct_solar_radiation_7_day_1h_mean_h', 
                              'cloudcover_total_lag_2_day_h', 
                              'cloudcover_total_lag_3_day_h', 
                              'cloudcover_total_lag_7_day_h', 
                              'cloudcover_total_7_day_mean_h',
                              'diffuse_radiation_lag_2_day_h',
                              'diffuse_radiation_lag_3_day_h',
                              'diffuse_radiation_lag_7_day_h',
                              'diffuse_radiation_7_day_1h_mean_h',
                              # 'shortwave_radiation_lag_2_day_h',
                              # 'shortwave_radiation_lag_7_day_h',
                             ]]

In [8]:
historical_weather_train = pd.read_csv(comp_path + "/historical_weather.csv")
historical_weather_train['datetime'] = pd.to_datetime(historical_weather_train['datetime'])

In [9]:
forecast_weather_train = pd.read_csv(comp_path + "/forecast_weather.csv")
forecast_weather_train['forecast_datetime'] = pd.to_datetime(forecast_weather_train['forecast_datetime'])

In [10]:
weather_features = [
    'temperature',
#     "dewpoint", 
#    "cloudcover_high", 
    "cloudcover_low", 
    "cloudcover_mid", 
    "cloudcover_total", 
    "10_metre_u_wind_component",
    "10_metre_v_wind_component",
#     "10_metre_u_wind_component_pos",
#     "10_metre_v_wind_component_pos",
#     "10_metre_u_wind_component_neg",
#     "10_metre_v_wind_component_neg",
#     "wind_speed",
    "direct_solar_radiation", 
    "surface_solar_radiation_downwards", 
#     "snowfall",
    'total_precipitation',
   ]


In [11]:
from functools import reduce
from itertools import chain

def get_weighted_forecast_by_county(forecast_weather_df, historical_weather_df,time_column='forecast_datetime', shift_feature=True, filter_hours_ahead=False):

    weather_condition_columns = [
        'temperature',
        'dewpoint',
        'cloudcover_high',
        'cloudcover_low',
        'cloudcover_mid',
        'cloudcover_total',
        '10_metre_u_wind_component',
        '10_metre_v_wind_component',
        'direct_solar_radiation',
        'surface_solar_radiation_downwards',
        'snowfall', 
        'total_precipitation',
        # 'temperature_bias',
        # 'direct_solar_radiation_bias',
    ]
    

    point_features= [
        'temperature', 
        'dewpoint', 
        'cloudcover_high',
        "cloudcover_low", 
        "cloudcover_mid", 
        "cloudcover_total", 
        "10_metre_u_wind_component",
        "10_metre_v_wind_component",
    ]

    cumulative_features= [
        'direct_solar_radiation', 
        'total_precipitation', 
        'surface_solar_radiation_downwards',
        'snowfall',
    ]

    new_df = forecast_weather_df.copy()
    if filter_hours_ahead:
        new_df = new_df[
            (new_df['hours_ahead'] >= 22)
            * (new_df['hours_ahead'] <= 45)
        ]
    new_df[['latitude', 'longitude']] = new_df[['latitude', 'longitude']].astype(np.float64).round(1)
    
    
    # The forecast data served during the test phase may have duplicate/missing coordinate pairs
    # Need to reindex the dataframe to avoid errors
    coors = historical_weather_train[['latitude', 'longitude']].set_index(['latitude', 'longitude']).sort_index().index.unique()
    timestamps = new_df[time_column].unique()
    
    idx = [[ts] + [*coor] for ts in timestamps for coor in coors]
#     idx = [[ts] + [*coor] for ts in timestamps.to_pandas() for coor in coors.to_pandas()]
    reidx = pd.DataFrame(idx, columns=['datetime', 'latitude', 'longitude']).set_index(['datetime', 'latitude', 'longitude']).index
    
    weather_condition_arr = new_df.sort_values(
                            [time_column, 'latitude', 'longitude']
                            ).groupby(
                                [time_column, 'latitude', 'longitude']
                                )[weather_condition_columns].mean()
    
    weather_condition_arr = weather_condition_arr.reindex(reidx).values.reshape(-1, 112, len(weather_condition_columns))
    
    
    new_index = new_df.sort_values([time_column]).groupby(
                            [time_column]
                            )[weather_condition_columns].mean().index.drop_duplicates()


    new_index = np.repeat(new_index, 16)
    
    
    weather_values_arr = (np.swapaxes(weather_condition_arr, 1, 2)  @ county_coordinate_weights.set_index('county_id').values.T)
    weather_values_arr = np.swapaxes(weather_values_arr, 1, 2).reshape(-1, len(weather_condition_columns))
    
    
    
    weather_by_county = pd.DataFrame(weather_values_arr, columns=weather_condition_columns, index=new_index)
    weather_by_county['county'] = np.array(np.arange(16).tolist() * (weather_values_arr.shape[0]//16) )
    
    weather_by_county.reset_index(inplace=True)
    weather_by_county['datetime'] = weather_by_county[time_column]
    
    
    
    weather_by_county = weather_by_county.set_index(['county', 'datetime']).sort_index()
    # weather_by_county['direct_solar_radiation'] = weather_by_county['direct_solar_radiation'].clip(0)
    
    if shift_feature:
#         weather_by_county[cumulative_features] = weather_by_county[cumulative_features].shift(-1)
        weather_by_county[point_features] = (weather_by_county[point_features].shift(1) + weather_by_county[point_features]) / 2
        
    
    lag_weather_features = ['direct_solar_radiation', 
                            'temperature',
                            'cloudcover_low',
#                             'cloudcover_total',
                            'dewpoint',
                            'total_precipitation',
                            'surface_solar_radiation_downwards',
                           # '10_metre_u_wind_component',
                           # '10_metre_v_wind_component',
                           ]
    
    for lag_day in [2, 3, 7, 14]:
        weather_by_county[[f"{ft}_lag_{lag_day}_day" for ft in lag_weather_features]] = weather_by_county[lag_weather_features].shift(24*lag_day)
        weather_by_county[[f"{ft}_{lag_day}_day_diff" for ft in lag_weather_features]] = weather_by_county[lag_weather_features].values  - weather_by_county[[f"{ft}_lag_{lag_day}_day" for ft in lag_weather_features]].values 
    
    
    special_lag_weather_features = ['direct_solar_radiation', 'temperature']

    # avoid fragmented dataframe
    weather_by_county = weather_by_county.copy()

    for lag_day in [9]:
        weather_by_county[[f"{ft}_lag_{lag_day}_day" for ft in special_lag_weather_features]] = weather_by_county[special_lag_weather_features].shift(24*lag_day)
        weather_by_county[[f"{ft}_{lag_day}_day_diff" for ft in special_lag_weather_features]] = weather_by_county[special_lag_weather_features].values  - weather_by_county[[f"{ft}_lag_{lag_day}_day" for ft in special_lag_weather_features]].values 
    
    for lag_day in [2, 3, 7]:
        weather_by_county[[f"{ft}_lag_{lag_day}_day_1h" for ft in special_lag_weather_features]] = weather_by_county[special_lag_weather_features].shift(24*lag_day+1)
        weather_by_county[[f"{ft}_{lag_day}_day_1h_diff" for ft in special_lag_weather_features]] = weather_by_county[special_lag_weather_features].values  - weather_by_county[[f"{ft}_lag_{lag_day}_day_1h" for ft in special_lag_weather_features]].values 

    weather_by_county[[f"{ft}_14_7_day_diff" for ft in special_lag_weather_features]] = weather_by_county[[f"{ft}_lag_14_day" for ft in special_lag_weather_features]].values   - weather_by_county[[f"{ft}_lag_7_day" for ft in special_lag_weather_features]].values
    weather_by_county[[f"{ft}_9_2_day_diff" for ft in special_lag_weather_features]] = weather_by_county[[f"{ft}_lag_9_day" for ft in special_lag_weather_features]].values   - weather_by_county[[f"{ft}_lag_2_day" for ft in special_lag_weather_features]].values

    
    # weather_by_county["direct_solar_radiation_7_day_1h_mean_diff"] = weather_by_county['direct_solar_radiation']  - np.mean([weather_by_county['direct_solar_radiation'].shift(24*lag_day+1) for lag_day in range(2, 9)], axis=0)
    # weather_by_county["surface_solar_radiation_downwards_7_day_1h_mean_diff"] = weather_by_county['surface_solar_radiation_downwards']  - np.mean([weather_by_county['surface_solar_radiation_downwards'].shift(24*lag_day+1) for lag_day in range(2, 9)], axis=0)

    
    weather_by_county["temperature_lag_1h"] = weather_by_county['temperature'].shift(1)
    
    
    weather_by_county[[f"{ft}_7_day_mean" for ft in lag_weather_features]] = sum([weather_by_county[lag_weather_features].shift(24*i) for i in range(2, 9)]) / 7
    weather_by_county[[f"{ft}_7_day_1h_mean" for ft in lag_weather_features]] = sum([weather_by_county[lag_weather_features].shift(24*i+1) for i in range(2, 9)]) / 7
    weather_by_county[[f"{ft}_8_14_day_mean" for ft in lag_weather_features]] = sum([weather_by_county[lag_weather_features].shift(24*i) for i in range(9, 16)]) / 7
    
    weather_by_county[[f"{ft}_7_day_mean_diff" for ft in lag_weather_features]] = weather_by_county[lag_weather_features].values  - weather_by_county[[f"{ft}_7_day_mean" for ft in lag_weather_features]].values 
    weather_by_county[[f"{ft}_7_day_1h_mean_diff" for ft in lag_weather_features]] = weather_by_county[lag_weather_features].values  - weather_by_county[[f"{ft}_7_day_1h_mean" for ft in lag_weather_features]].values 
    weather_by_county[[f"{ft}_8_14_day_mean_diff" for ft in lag_weather_features]] = weather_by_county[lag_weather_features].values  - weather_by_county[[f"{ft}_8_14_day_mean" for ft in lag_weather_features]].values 
    
    # weather_by_county[['snowfall_1_day_rolling_sum']] = weather_by_county[['snowfall']].reset_index().groupby(['county'])[['snowfall']].rolling(24*1).sum().values
    weather_by_county['snowfall_1_day_rolling_sum'] = weather_by_county['snowfall'].rolling(24*1).sum().values

    
    for i in [2, 7]:
        weather_by_county[[f'snowfall_1_day_rolling_sum_lag_{i}_day_diff']] = weather_by_county[['snowfall_1_day_rolling_sum']].shift(24*i).values - weather_by_county[['snowfall_1_day_rolling_sum']].values
    

    # weather_by_county['wind_speed'] = weather_by_county[['10_metre_u_wind_component', '10_metre_v_wind_component']].pow(2).sum(axis=1).pow(0.5)
    weather_by_county['temp_minus_dewpoint'] = weather_by_county['temperature'] - weather_by_county['dewpoint']
    weather_by_county["temp_above_cp"] = np.clip(weather_by_county['temperature'].values  - 14.6, 0, 200)
    weather_by_county["temp_below_cp"] = np.clip(14.6 - weather_by_county['temperature'].values, 0, 100)

    
    weather_by_county = weather_by_county.reset_index()
    
    
    weather_by_county = weather_by_county.merge(
        historical_weather_df,
        how="left",
        on=["county", "datetime",],
        )
    
    # Override with historical weather
    weather_by_county["direct_solar_radiation_7_day_mean_diff"] = weather_by_county['direct_solar_radiation'].values  - weather_by_county['direct_solar_radiation_7_day_mean_h'].values 
    weather_by_county["direct_solar_radiation_2_day_diff"] = weather_by_county['direct_solar_radiation'].values  - weather_by_county['direct_solar_radiation_lag_2_day_h'].values 
    weather_by_county["direct_solar_radiation_3_day_diff"] = weather_by_county['direct_solar_radiation'].values  - weather_by_county['direct_solar_radiation_lag_3_day_h'].values 
    weather_by_county["direct_solar_radiation_7_day_diff"] = weather_by_county['direct_solar_radiation'].values  - weather_by_county['direct_solar_radiation_lag_7_day_h'].values 

    weather_by_county["cloudcover_total_7_day_mean_diff"] = weather_by_county['cloudcover_total'].values  - weather_by_county['cloudcover_total_7_day_mean_h'].values 
    weather_by_county["cloudcover_total_2_day_diff"] = weather_by_county['cloudcover_total'].values  - weather_by_county['cloudcover_total_lag_2_day_h'].values 
    weather_by_county["cloudcover_total_3_day_diff"] = weather_by_county['cloudcover_total'].values  - weather_by_county['cloudcover_total_lag_3_day_h'].values 
    weather_by_county["cloudcover_total_7_day_diff"] = weather_by_county['cloudcover_total'].values  - weather_by_county['cloudcover_total_lag_7_day_h'].values 

    
    return weather_by_county

## Electricity & Gas Price

In [12]:
def process_elec_prices(process_elec_prices_df):
    
    new_df = process_elec_prices_df.copy()
    
    new_df.forecast_date = pd.to_datetime(new_df.forecast_date)
    new_df.origin_date = pd.to_datetime(new_df.origin_date)
    new_df['data_available_datetime'] = new_df.forecast_date + datetime.timedelta(days=1)
    
    
    new_df['data_available_date'] = new_df['data_available_datetime'].dt.date
#     new_df['data_available_date'] = new_df['data_available_datetime'].dt.floor("D")
    

    for i in range(1, 8):
        new_df[f'euros_per_mwh_lag_{i}_day'] = new_df['euros_per_mwh'].shift(24*i)

    euros_per_mwh_lag_7_day_mean = sum([new_df[f'euros_per_mwh_lag_{i}_day'] for i in range(1, 8)]) / 7
#     euros_per_mwh_lag_28_day_mean = sum([new_df['euros_per_mwh'].shift(24*i) for i in range(1, 29)]) / 28

    new_df[f'euros_per_mwh_lag_7_day_mean_diff'] = new_df['euros_per_mwh'] - euros_per_mwh_lag_7_day_mean
#     new_df[f'euros_per_mwh_lag_28_day_mean_diff'] = new_df['euros_per_mwh'] - euros_per_mwh_lag_28_day_mean

    new_df[f'euros_per_mwh_lag_7_day_diff'] = new_df['euros_per_mwh'] - new_df[f'euros_per_mwh_lag_7_day']
    new_df[f'euros_per_mwh_lag_2_day_diff'] = new_df['euros_per_mwh'] - new_df[f'euros_per_mwh_lag_2_day']


    daily_elec_prices = new_df.groupby(['data_available_date'])[['euros_per_mwh']].mean().reset_index()
    daily_elec_prices = daily_elec_prices.rename(columns={"euros_per_mwh": "daily_euros_per_mwh"})


    daily_elec_prices['daily_euros_per_mwh_lag_1_day'] = daily_elec_prices['daily_euros_per_mwh'].shift(1)
    # daily_elec_prices['daily_euros_per_mwh_lag_2_day'] = daily_elec_prices['daily_euros_per_mwh'].shift(2)
    # daily_elec_prices['daily_euros_per_mwh_lag_3_day'] = daily_elec_prices['daily_euros_per_mwh'].shift(3)
    daily_elec_prices['daily_euros_per_mwh_lag_7_day'] = daily_elec_prices['daily_euros_per_mwh'].shift(7)
    daily_elec_prices['daily_euros_per_mwh_rolling_7_day_mean'] =  daily_elec_prices['daily_euros_per_mwh'].rolling(7).mean()
    
    
    new_df = new_df.merge(
                        daily_elec_prices,
                        on=['data_available_date'],
                        how='left'
                    )
    
    return new_df

In [13]:
# def propcess_gas_prices(gas_prices_df):

#     new_df = gas_prices_df.copy()
#     new_df['forecast_date'] = pd.to_datetime(new_df['forecast_date']).dt.date
#     new_df['data_available_date'] = new_df.forecast_date + datetime.timedelta(days=1)
#     new_df['gas_mid_price'] = (new_df['lowest_price_per_mwh'] + new_df['highest_price_per_mwh']) /2
    
#     new_df['gas_mid_price_lag_1_day'] = new_df['gas_mid_price'].shift(1)

#     new_df['gas_mid_price_lag_7_day'] = new_df['gas_mid_price'].shift(7)
#     new_df['gas_mid_price_rolling_7_day_mean'] =  new_df['gas_mid_price'].rolling(7).mean()
    
#     return new_df
    

## Merge dataframe

In [14]:
%%time

train = train.merge(
    process_elec_prices(elec_prices)[[
        "euros_per_mwh", 
#          "daily_euros_per_mwh",
#          'daily_euros_per_mwh_lag_1_day',
#          'daily_euros_per_mwh_lag_7_day',
#          'daily_euros_per_mwh_rolling_7_day_mean',
        "data_available_datetime",
        "euros_per_mwh_lag_7_day_mean_diff",
#         "euros_per_mwh_lag_28_day_mean_diff",
        "euros_per_mwh_lag_7_day_diff",
        "euros_per_mwh_lag_2_day_diff",
        ]],
    how="left",
    right_on="data_available_datetime",
    left_on="datetime",
   )

# train = train.merge(
#     propcess_gas_prices(gas_prices_train)[[
#         "data_available_date",
#         "gas_mid_price",
#         'gas_mid_price_lag_1_day',
#         'gas_mid_price_lag_7_day',
#         'gas_mid_price_rolling_7_day_mean',
#         ]],
#     how="left",
#     right_on="data_available_date",
#     left_on="date",
#    )



train = train.merge(
        client_train.drop(["data_block_id", "date"], axis=1),  
        left_on=["county", "is_business", "product_type", "date"], 
        right_on=["county", "is_business", "product_type", "available_date"], 
        how="left")

train = train.merge(
    get_weighted_forecast_by_county(
        forecast_weather_train, 
        get_weighted_historical_weather_by_county(historical_weather_train), 
        filter_hours_ahead=False
    ),
    how="left",
    on=["county", "datetime",],
    )


CPU times: user 13 s, sys: 1.52 s, total: 14.5 s
Wall time: 14.5 s


In [15]:
def fill_missing_time(df, start_time=None, end_time=None, inference=False):
    
    if start_time is None:
        start_time = df['datetime'].min()
        
    if end_time is None:
        end_time = df['datetime'].max()
        
    df_array = np.empty((16, 2, 4, 2), dtype=object)
    idx = pd.date_range(start_time, end_time, freq='H')
    
    for county in range(16):
        for is_business in [0, 1]:
            for product_type in [0, 1, 2, 3]:
                for is_consumption in [0, 1]:

#                     print(county, is_business, product_type, is_consumption, end="")
                    try:
                        temp_df = df[
                                    (df['county']==county) *
                                    (df['is_business']==is_business) *
                                    (df['product_type']==product_type) *
                                    (df['is_consumption']==is_consumption)
                                    ].set_index('datetime')
                        length = len(temp_df)
                        
                        if inference and length==0:
                            continue

                        temp_df = temp_df.reindex(
                            idx, 
#                             method='nearest',
                        )
                        temp_df[["county", "is_business", "product_type", "is_consumption"]] = [county, is_business, product_type, is_consumption]
                        df_array[county, is_business, product_type, is_consumption] = temp_df

                        
#                         print(len(df_array[county, is_business, product_type, is_consumption]))
                    except:
#                         print(0)
                        pass

                for is_consumption in [0, 1]:
                    if df_array[county, is_business, product_type, 1-is_consumption] is None:
                        continue
                    df_array[county, is_business, product_type, is_consumption]['consumption_counterpart'] = df_array[county, is_business, product_type, 1-is_consumption]['target']
            
                
    new_df = pd.concat([e for e in df_array.flatten() if e is not None])
    new_df.index.names = ['datetime']
    new_df = new_df.reset_index()
    
    new_df['date'] = new_df['datetime'].dt.date 
    
#     new_df = new_df.set_index(['county', 'is_business', 'product_type', 'is_consumption', 'datetime']).sort_index(ascending=True)
    return new_df, df_array

In [16]:
import sys
sys.getsizeof(train)

2696449296

In [17]:
non_float_features = ['row_id', 'prediction_unit_id', 'day']

In [18]:
for col in train.columns:
    if col not in non_float_features:
        if train[col].dtype == float:
            train[col] = train[col].astype(np.float32)
print(sys.getsizeof(train))

1509658320


In [19]:
%%time
train, df_arr = fill_missing_time(train)

CPU times: user 13.3 s, sys: 1.49 s, total: 14.8 s
Wall time: 14.8 s


## Append Lag features

In [20]:
%%time

train = train.set_index(['county', 'is_business', 'product_type', 'is_consumption', 'datetime']).sort_index(ascending=True)

base_lag = 48


grouped_eic_count = train.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['eic_count']
# train['eic_count_2_day_diff'] = grouped_eic_count.shift(0).fillna(0) - grouped_eic_count.shift(24*0).fillna(0)
# train['eic_count_7_day_diff'] = grouped_eic_count.shift(0).fillna(0) - grouped_eic_count.shift(24*5).fillna(0)
# train['eic_count_7_day_mean_diff'] = grouped_eic_count.shift(0).fillna(0) - np.mean([grouped_eic_count.shift(24*i).fillna(0) for i in range(0, 7)], axis=0)

# train['eic_count_2_day_diff'] = grouped_eic_count.shift(0) - grouped_eic_count.shift(24*0)
train['eic_count_7_day_diff'] = grouped_eic_count.shift(0) - grouped_eic_count.shift(24*5)
# train['eic_count_14_day_diff'] = grouped_eic_count.shift(0) - grouped_eic_count.shift(24*(5+7))
train['eic_count_7_day_mean_diff'] = grouped_eic_count.shift(0) - np.mean([grouped_eic_count.shift(24*i) for i in range(0, 7)], axis=0)

grouped_installed_capacity = train.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['installed_capacity']
# train['installed_capacity_2_day_diff'] = grouped_installed_capacity.shift(0).fillna(0) - grouped_installed_capacity.shift(24*0).fillna(0)
# train['installed_capacity_7_day_diff'] = grouped_installed_capacity.shift(0).fillna(0) - grouped_installed_capacity.shift(24*5).fillna(0)
# train['installed_capacity_7_day_mean_diff'] = grouped_installed_capacity.shift(0).fillna(0) - np.mean([grouped_installed_capacity.shift(24*i).fillna(0) for i in range(0, 7)], axis=0)

# train['installed_capacity_2_day_diff'] = grouped_installed_capacity.shift(0) - grouped_installed_capacity.shift(24*0)
train['installed_capacity_7_day_diff'] = grouped_installed_capacity.shift(0) - grouped_installed_capacity.shift(24*5)
# train['installed_capacity_14_day_diff'] = grouped_installed_capacity.shift(0) - grouped_installed_capacity.shift(24*(5+7))
train['installed_capacity_7_day_mean_diff'] = grouped_installed_capacity.shift(0) - np.mean([grouped_installed_capacity.shift(24*i) for i in range(0, 7)], axis=0)


### scale target by installed_capacity
# train['target_original'] = train['target']
# train['target'] = train['target'] / train['installed_capacity']
###


grouped_target = train.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target']

for i in range(2, 22):
    train[f'lag_{i}_day'] = grouped_target.shift(24*i).fillna(0)
    
for i in range(2, 9):
# for i in [2,3,7]:
    train[f'lag_{i}_day_1h'] = grouped_target.shift(24*i+1).fillna(0)

train['lag_7_day_mean'] = np.mean([train[f'lag_{i}_day'] for i in range(2, 9)], axis=0)
train['lag_7_day_1h_mean'] = np.mean([train[f'lag_{i}_day_1h'] for i in range(2, 9)], axis=0)

train['lag_8_14_day_mean'] = np.mean([train[f'lag_{i}_day'] for i in range(9, 16)], axis=0)

train['lag_2_day_1h'] = grouped_target.shift(48+1)
train['lag_3_day_1h'] = grouped_target.shift(72+1)
# train['lag_4_day_1h'] = grouped_target.shift(96+1)
train['lag_7_day_1h'] = grouped_target.shift(24*7+1)
train['lag_7_day_m1h'] = grouped_target.shift(24*7-1)

rolling_target = train.reset_index().groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target'].rolling(24)

train['rolling_mean_24'] = rolling_target.mean().shift(base_lag).values

# train['rolling_mean_24x7'] = train.reset_index().groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target'].rolling(24*7).mean().shift(base_lag).values
train['rolling_std_24'] = rolling_target.std().shift(base_lag).values
train['rolling_kurt_24'] = rolling_target.kurt().shift(base_lag).values
train['rolling_skew_24'] = rolling_target.skew().shift(base_lag).values



train['rolling_mean_24_lag_7_day'] = rolling_target.mean().shift(24*7).values
train['rolling_mean_24_lag_9_day'] = rolling_target.mean().shift(24*9).values
train['ratio_rolling_mean_lag_9_7_day'] = (train['rolling_mean_24_lag_9_day'] + 10) /(train['rolling_mean_24_lag_7_day'] + 10)

# train['rolling_std_24_lag_7_day'] = train.reset_index().groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target'].rolling(24).std().shift(24*7).values
# train['rolling_kurt_24_lag_7_day'] = train.reset_index().groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target'].rolling(24).kurt().shift(24*7).values
# train['rolling_skew_24_lag_7_day'] = train.reset_index().groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target'].rolling(24).skew().shift(24*7).values


grouped_consumption_counterpart = train.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['consumption_counterpart']


# for i in range(2, 9):
for i in [2, 3, 7, 14]:
    train[f'lag_{i}_day_consumption_counterpart'] = grouped_consumption_counterpart.shift(24*i)
    
train[f'lag_2_day_1h_consumption_counterpart'] = grouped_consumption_counterpart.shift(24*2+1)

train['lag_7_day_mean_consumption_counterpart'] = np.mean([grouped_consumption_counterpart.shift(24*i) for i in range(2, 9)], axis=0)

# train['rolling_mean_24_consumption_counterpart'] = grouped_consumption_counterpart.rolling(24).mean().shift(base_lag).values

train['ratio_lag_9_2_day'] = (train['lag_9_day'] + 10) /(train['lag_2_day'] + 10)
train['ratio_lag_10_3_day'] = (train['lag_10_day'] + 10) /(train['lag_3_day'] + 10)
train['ratio_lag_14_7_day'] = (train['lag_14_day'] + 10) /(train['lag_7_day'] + 10)
train['ratio_lag_9_7_day'] = (train['lag_9_day'] + 10) /(train['lag_7_day'] + 10)


train['ratio_lag_21_7_day'] = (train['lag_21_day'] + 10) /(train['lag_7_day'] + 10)
# train['ratio_lag_16_14_day'] = (train['lag_16_day'] + 10) /(train['lag_14_day'] + 10)

# train['ratio_lag_9_2_day'] = train['lag_9_day'] - train['lag_2_day'] 
# train['ratio_lag_10_3_day'] = train['lag_10_day'] - train['lag_3_day'] 
# train['ratio_lag_14_7_day'] = train['lag_14_day'] - train['lag_7_day'] 
# train['ratio_lag_9_7_day'] = train['lag_9_day'] - train['lag_7_day'] 

# train['ratio_lag_9_7_day'] = train['lag_9_day'] - train['lag_7_day'] 

train = train.reset_index()


grouped_target_all = train[['datetime', 'is_business', 'is_consumption', 'target']].groupby(['is_business', 'is_consumption','datetime'])[['target']].sum()
grouped_target_all['target_all_lag_2_day'] = grouped_target_all['target'].shift(24*2)
grouped_target_all['target_all_lag_7_day'] = grouped_target_all['target'].shift(24*7)
del grouped_target_all['target']

grouped_consumption_counterpart_all = train[['datetime', 'county', 'is_business', 'is_consumption', 'consumption_counterpart']].groupby(['is_business', 'is_consumption','datetime'])[['consumption_counterpart']].sum()
grouped_consumption_counterpart_all['consumption_counterpart_all_lag_2_day'] = grouped_consumption_counterpart_all['consumption_counterpart'].shift(24*2)
grouped_consumption_counterpart_all['consumption_counterpart_all_lag_3_day'] = grouped_consumption_counterpart_all['consumption_counterpart'].shift(24*3)
grouped_consumption_counterpart_all['consumption_counterpart_all_lag_7_day'] = grouped_consumption_counterpart_all['consumption_counterpart'].shift(24*7)
del grouped_consumption_counterpart_all['consumption_counterpart']



train = train.merge(
        grouped_target_all,
        on=["datetime", "is_business", 'is_consumption'], 
        how="left")

train = train.merge(
        grouped_consumption_counterpart_all,
        on=["datetime", "is_business", 'is_consumption'], 
        how="left")


CPU times: user 15.8 s, sys: 9.09 s, total: 24.8 s
Wall time: 24.9 s


In [21]:
national_holidays =[
                    [2021, 1, 1],
                    [2021, 2, 24],
                    [2021, 4, 2],
                    [2021, 4, 4],
                    [2021, 5, 1],
                    [2021, 5, 23],
                    [2021, 6, 23],
                    [2021, 6, 24],
                    [2021, 8, 20],
                    [2021, 12, 24],
                    [2021, 12, 25],
                    [2021, 12, 26],


                    [2022, 1, 1],
                    [2022, 2, 24],
                    [2022, 4, 15],
                    [2022, 4, 17],
                    [2022, 5, 1],
                    [2022, 6, 5],
                    [2022, 6, 23],
                    [2022, 6, 24],
                    [2022, 8, 20],
                    [2022, 12, 24],
                    [2022, 12, 25],
                    [2022, 12, 26],

                    [2023, 1, 1],
                    [2023, 2, 24],
                    [2023, 4, 7],
                    [2023, 4, 9],
                    [2023, 5, 1],
                    [2023, 5, 28],
                    [2023, 6, 23],
                    [2023, 6, 24],
                    [2023, 8, 20],
                    [2023, 12, 24],
                    [2023, 12, 25],
                    [2023, 12, 26],

                    [2024, 1, 1],
                    [2024, 2, 24],
                    [2024, 3, 29],
                    [2024, 3, 31],
                    [2024, 5, 1],
                    [2024, 5, 19],
                    [2024, 6, 23],
                    [2024, 6, 24],
                    [2024, 8, 20],
                    [2024, 12, 24],
                    [2024, 12, 25],
                    [2024, 12, 26],
]

In [22]:
import datetime
national_holidays_dt = [pd.to_datetime('-'.join(str(n) for n in holiday)).date() for holiday in national_holidays]

def append_daytype(train_df):
    train_df['days_in_month'] = train_df['datetime'].dt.days_in_month

    train_df['inner_weekday'] = (train_df['datetime'].dt.dayofweek >= 1) * (train_df['datetime'].dt.dayofweek <= 3) # Tuesday to Thursday
    train_df['holiday'] = (train_df['datetime'].dt.dayofweek >= 5) * (train_df['datetime'].dt.dayofweek <= 6) # Saturday and Sunday
    train_df['pre_holiday'] = train_df['datetime'].dt.dayofweek == 4
    train_df['post_holiday'] = train_df['datetime'].dt.dayofweek == 0
    
    train_df['national_holiday'] = train_df['datetime'].dt.date.isin(national_holidays_dt)
    train_df['pre_national_holiday'] = train_df['datetime'].dt.date.isin(d-datetime.timedelta(days=1) for d in national_holidays_dt)
    train_df['post_national_holiday'] = train_df['datetime'].dt.date.isin(d+datetime.timedelta(days=1) for d in national_holidays_dt)
    
    train_df['daytype'] = None
    train_df.loc[train_df['inner_weekday'], 'daytype'] = 'inner_weekday'
    train_df.loc[train_df['pre_holiday'] + train_df['pre_national_holiday'], 'daytype'] = 'pre_holiday'
    train_df.loc[train_df['post_holiday'] + train_df['post_national_holiday'], 'daytype'] = 'post_holiday'
    train_df.loc[train_df['holiday'] + train_df['national_holiday'], 'daytype'] = 'holiday'
    
    # train_df.loc[train_df['inner_weekday'], 'daytype'] = 0
    # train_df.loc[train_df['pre_holiday'] + train_df['pre_national_holiday'], 'daytype'] = 1
    # train_df.loc[train_df['post_holiday'] + train_df['post_national_holiday'], 'daytype'] = 2
    # train_df.loc[train_df['holiday'] + train_df['national_holiday'], 'daytype'] = 3
    
    

    # train_df['dayofweek'] = train_df['dayofweek'].astype(dtype)
    # train_df['dayofweek_lag_2_day'] = train_df['dayofweek'].shift(24*2)
    # train_df['dayofweek_lag_3_day'] = train_df['dayofweek'].shift(24*3)
    # train_df['dayofweek_lag_7_day'] = train_df['dayofweek'].shift(24*7)
    
    fts_to_remove = ['inner_weekday', 'holiday', 'pre_holiday', 'post_holiday', 'national_holiday', 'pre_national_holiday', 'post_national_holiday']
    for ft in fts_to_remove:
        del train_df[ft]
        
    
    train_df['day_category'] = train_df['dayofweek']
    
    train_df.loc[
        (train_df['daytype'] == 'holiday'),'day_category'
    ] = 5
    
    train_df.loc[
        (train_df['daytype'] == 'holiday')
        * (train_df['dayofweek'].isin([0, 6])), 'day_category'
    ] = 6
    
    
    train_df.loc[
        (train_df['daytype'] == 'pre_holiday')
        * (train_df['dayofweek'].isin([0, 1, 2, 3, 4])), 'day_category'
    ]= 4

    train_df['dayofweek'] = train_df['dayofweek'].astype(dtype)
    train_df['day_category'] = train_df['day_category'].astype(dtype)


    train_df['daytype'] = train_df['daytype'].astype(dtype)
    train_df['daytype_lag_2_day'] = train_df['daytype'].shift(24*2)
    train_df['daytype_lag_3_day'] = train_df['daytype'].shift(24*3)
    train_df['daytype_lag_7_day'] = train_df['daytype'].shift(24*7)


    train_df['day_category_lag_2_day'] = train_df['day_category'].shift(24*2)
    train_df['day_category_lag_3_day'] = train_df['day_category'].shift(24*3)
    train_df['day_category_lag_7_day'] = train_df['day_category'].shift(24*7)
    
    return train_df

In [23]:
def append_school_holiday(train_df):
    """
    school holidays:
    
    Feb 22 - Feb 28,
    Apr 19 - Apr 25,
    Jun 14 - Aug 31,
    Oct 25 - Oct 31,
    Dec 23, 2021 - Jan 9, 2022,

    Feb 28 - Mar 6,
    Apr 25 - May 1,
    Jun 14 - Aug 31,
    Oct 24 - Oct 30,
    Dec 22, 2022 - Jan 8, 2023,

    Feb 27 - Mar 5,
    Apr 24 - Apr 30,
    Jun 14 - Aug 31,
    Oct 23 - Oct 29,
    Dec 21, 2023 - Jan 7, 2024,
    """
    
    school_holiday_range = [
        ["2021-02-22", "2021-02-28"],
        ["2021-04-19", "2021-04-25"],
        ["2021-06-14", "2021-08-31"],
        ["2021-10-25", "2021-10-31"],
        ["2021-12-23", "2022-01-09"],

        ["2022-02-23", "2022-03-06"],
        ["2022-04-25", "2022-05-01"],
        ["2022-06-14", "2022-08-31"],
        ["2022-10-24", "2022-10-30"],    
        ["2022-12-22", "2023-01-08"],

        ["2023-02-27", "2023-03-05"],
        ["2023-04-24", "2023-04-30"],
        ["2023-06-14", "2023-08-31"],
        ["2023-10-23", "2023-10-29"],    
        ["2023-12-21", "2024-01-07"],

        ["2024-02-26", "2024-03-03"],
        ["2024-04-22", "2024-04-28"],
        ["2024-06-13", "2024-08-31"],]

    school_holiday_masks = [(train_df['datetime'] >= pd.to_datetime(start_date)) * (train_df['datetime'] <= pd.to_datetime(end_date)) 
                            for start_date, end_date in school_holiday_range]
    
    train_df['school_holiday'] = np.sum(school_holiday_masks,axis=0).astype(bool)
    
    return train_df

In [24]:
%%time

# dtype = str
dtype = "category"



train["month"]  = train["datetime"].dt.month
train['day'] = train['datetime'].dt.day
train['yearmonth'] = train['datetime'].dt.year.astype(str) + train['datetime'].dt.month.astype(str)
train['yearmonthday'] = train['datetime'].dt.year.astype(str) +train['datetime'].dt.month.astype(str) + train['datetime'].dt.day.astype(str)
train['dayofweek'] = train['datetime'].dt.dayofweek
train = append_daytype(train)

# train['dayofweek'] = train['dayofweek'].astype(dtype)
# train['day_category'] = train['day_category'].astype(dtype)
# train = append_school_holiday(train)

hours_in_day = 24
train['hour'] = train['datetime'].dt.hour
train['sin_hour'] = np.sin(2*np.pi*train.hour/hours_in_day)
train['cos_hour'] = np.cos(2*np.pi*train.hour/hours_in_day)
train['hour'] = train['hour'].astype(dtype)


months_in_year = 12
train['sin_month'] = np.sin(2*np.pi*train.month/months_in_year)
train['cos_month'] = np.cos(2*np.pi*train.month/months_in_year)
train['month'] = train['month'].astype(dtype)


# days_in_year = 365
# train['sin_day'] = np.sin(2*np.pi*train['datetime'].dt.day_of_year/days_in_year)
# train['cos_day'] = np.cos(2*np.pi*train['datetime'].dt.day_of_year/days_in_year)


train['county'] = train['county'].astype(dtype)
train['product_type'] = train['product_type'].astype(dtype)

CPU times: user 11.6 s, sys: 1.26 s, total: 12.8 s
Wall time: 12.8 s


In [25]:
train = train[~train['target'].isna()]

print(int(100*len(train)/2018352),"% of dataset")
train.head()

99 % of dataset


,county,is_business,product_type,is_consumption,datetime,target,data_block_id,row_id,prediction_unit_id,date,...,daytype_lag_3_day,daytype_lag_7_day,day_category_lag_2_day,day_category_lag_3_day,day_category_lag_7_day,hour,sin_hour,cos_hour,sin_month,cos_month
30624,0,0,1,0,2021-09-01 00:00:00,0.713,0.0,0.0,0.0,2021-09-01,...,post_holiday,inner_weekday,1,0,3,0,0.000000,1.000000,-1.0,-1.836970e-16
30625,0,0,1,0,2021-09-01 01:00:00,1.132,0.0,122.0,0.0,2021-09-01,...,post_holiday,inner_weekday,1,0,3,1,0.258819,0.965926,-1.0,-1.836970e-16
30626,0,0,1,0,2021-09-01 02:00:00,0.490,0.0,244.0,0.0,2021-09-01,...,post_holiday,inner_weekday,1,0,3,2,0.500000,0.866025,-1.0,-1.836970e-16
30627,0,0,1,0,2021-09-01 03:00:00,0.496,0.0,366.0,0.0,2021-09-01,...,post_holiday,inner_weekday,1,0,3,3,0.707107,0.707107,-1.0,-1.836970e-16
30628,0,0,1,0,2021-09-01 04:00:00,0.149,0.0,488.0,0.0,2021-09-01,...,post_holiday,inner_weekday,1,0,3,4,0.866025,0.500000,-1.0,-1.836970e-16


In [26]:
raw_features = [
                "county",
                "is_business",
                "product_type",
                "is_consumption",
                "dayofweek",
                "daytype",
                "daytype_lag_3_day",
                # "month",
#                 "sin_month",
#                 "cos_month",
#                 "hour",
                "sin_hour",
                "cos_hour",
                "lag_2_day",
                "lag_3_day",
                "lag_4_day",
                "lag_7_day_mean",
                "lag_5_day",
                "lag_6_day",
                "lag_7_day",

                # "lag_14_day",
                "lag_2_day_1h",
#                 "lag_3_day_1h",
#                 "lag_4_day_1h",
                
                "rolling_mean_24",
                # "rolling_mean_24x7",
                "rolling_std_24",
                "lag_2_day_consumption_counterpart",
                # "lag_2_day_1h_consumption_counterpart",
                # "lag_3_day_consumption_counterpart",
#                 "lag_4_day_consumption_counterpart",
#                 "lag_5_day_consumption_counterpart",
#                 "lag_6_day_consumption_counterpart",
#                 "lag_7_day_consumption_counterpart",
                "lag_7_day_mean_consumption_counterpart",
                # "rolling_mean_24_consumption_counterpart",
#                 "euros_per_mwh",
#                 "daily_euros_per_mwh", 
#                 'daily_euros_per_mwh_lag_1_day',
#                 'daily_euros_per_mwh_lag_7_day',
#                 'daily_euros_per_mwh_rolling_7_day_mean',
                # Gas price
#                 "gas_mid_price",
#                 'gas_mid_price_lag_1_day',
#                 'gas_mid_price_lag_7_day',
#                 'gas_mid_price_rolling_7_day_mean',
                'snowfall_1_day_rolling_sum',
#                 "school_holiday",
#                 'wind_speed',
    
                'eic_count_7_day_mean_diff',
                'installed_capacity_7_day_mean_diff',
    
                'cloudcover_total_7_day_mean_diff',
                # "sin_day",
                # "cos_day",

                # "temperature_bias",
                # '10_metre_u_wind_component_7_day_mean_diff',
                # '10_metre_v_wind_component_7_day_mean_diff',
                # 'direct_solar_radiation_2_day_1h_diff',
                # "euros_per_mwh_lag_7_day_mean_diff",

                # '10_metre_u_wind_component_lag_2_day',
                # '10_metre_v_wind_component_lag_2_day',
                # 'lag_7_day_1h_mean',
                # 'direct_solar_radiation_7_day_1h_mean_diff',
                # 'surface_solar_radiation_downwards_7_day_1h_mean_diff',
                # 'wind_speed',
                # 'diffuse_radiation_7_day_mean_h',

                # 'lag_7_day_1h',
                # 'direct_solar_radiation_7_day_1h_diff', 
                # 'temperature_7_day_1h_diff', 
                'direct_solar_radiation_2_day_1h_diff',
                'ratio_lag_9_2_day'
                
               ] # + [f'product_type_embed_{i}' for i in range(8)] #+ [f'county_embed_{i}' for i in range(2)]


In [27]:
train.reset_index(drop=True, inplace=True)
# X = train.loc[train[train['hour'] == 0].index][features]
# Y = np.array([train['target'][train[train['hour'] == 0].index + i].values for i in range(24)]).T
# X.shape, Y.shape

In [28]:
def select_features(raw_features, weather_features, client_features, is_consumption, target='target', verbose=0):
    
    features = raw_features + weather_features + client_features
    features.remove('is_consumption')
#     features.remove('is_business')
    
    fts_to_remove = []
    fts_to_add = []
    
    
    if is_consumption==0:

        fts_to_add +=[
            # 'consumption_counterpart_all_type_lag_2_day',
            # 'consumption_counterpart_all_type_lag_7_day',
            # 'consumption_counterpart_all_lag_2_day',
            'consumption_counterpart_all_lag_7_day',

            # 'ratio_lag_9_2_day',

        ]
        fts_to_remove += [
            "lag_4_day",
            # "lag_5_day",
            # "lag_6_day",
#             "lag_7_day",
            
#             "lag_2_day_consumption_counterpart",
#             "lag_7_day_consumption_counterpart",
            # "is_business",
        ]
    elif is_consumption==1:
        fts_to_remove += [
#             'cloudcover_low',
            'cloudcover_mid',
            'cloudcover_total',
            '10_metre_u_wind_component',
            '10_metre_v_wind_component',
            
#             "wind_speed",
#             "dewpoint",
            # "lag_5_day",
            # "lag_6_day",
#             "sin_hour"
#             "total_precipitation",
            # "sin_hour",
            # "cos_hour",
        ]
        
        fts_to_add += [
            'hour',
            "rolling_kurt_24",
            "rolling_skew_24",
            "lag_3_day_consumption_counterpart",
            # "rolling_mean_24x7",
#             'school_holiday',
            # "lag_7_day_1h",
            # "lag_7_day_m1h",

            'temperature_2_day_1h_diff',
            # "target_all_type",
            # 'target_all_type_lag_2_day',
            # 'target_all_type_lag_7_day',
            'target_all_lag_2_day',
            'target_all_lag_7_day',
            
            # "ratio_lag_14_7_day",

            'ratio_lag_9_2_day',
        ]
        
    else:
        raise ValueError
        
        
    if is_consumption==0:
        fts_to_add += [
            'temp_minus_dewpoint',
            
            'direct_solar_radiation_7_day_mean_diff', 
#             'temperature_7_day_mean_diff', 
            'cloudcover_low_7_day_mean_diff',
#             'dewpoint_7_day_mean_diff',
            'total_precipitation_7_day_mean_diff',
            'surface_solar_radiation_downwards_7_day_mean_diff',
            
            
            'direct_solar_radiation_2_day_diff', 
#             'temperature_2_day_diff', 
            'cloudcover_low_2_day_diff', 
#             'dewpoint_2_day_diff', 
            'total_precipitation_2_day_diff', 
            'surface_solar_radiation_downwards_2_day_diff',
            
            # 'direct_solar_radiation_3_day_diff', 
            # # 'temperature_3_day_diff', 
            # 'cloudcover_low_3_day_diff', 
            # # 'dewpoint_3_day_diff', 
            # 'total_precipitation_3_day_diff', 
            # 'surface_solar_radiation_downwards_3_day_diff',
            
#             'wind_speed',
#             'daily_euros_per_mwh_lag_1_day',
#             'daily_euros_per_mwh_lag_7_day', 
#             'daily_euros_per_mwh_rolling_7_day_mean',
                
                # Gas price
#                 'gas_mid_price_lag_1_day',
#                 'gas_mid_price_lag_7_day',
#                 'gas_mid_price_rolling_7_day_mean',
#             'euros_per_mwh',
            "lag_8_14_day_mean",
            'snowfall_1_day_rolling_sum_lag_2_day_diff',
            # 'snowfall_1_day_rolling_sum_lag_7_day_diff',

            # 'direct_solar_radiation_3_day_diff', 
            # 'surface_solar_radiation_downwards_3_day_diff',
            # 'direct_solar_radiation_bias',
            # 'direct_solar_radiation_3_day_diff', 
        ]


        if target in ['lag_2_day_diff', 'lag_2_day_diff_neg']:
            fts_to_add += [
                'diffuse_radiation_lag_2_day_h',
                # 'shortwave_radiation_lag_2_day_h',

                'consumption_counterpart_all_lag_2_day',
            ]


        if target in ['lag_3_day_diff', 'lag_3_day_diff_neg']:
            fts_to_add += [
                'diffuse_radiation_lag_3_day_h',
                # 'shortwave_radiation_lag_2_day_h',

                'consumption_counterpart_all_lag_3_day',
            ]

        if target in ['lag_7_day_diff', 'lag_7_day_diff_neg']:
            fts_to_add += [
                'diffuse_radiation_lag_7_day_h',
                # 'shortwave_radiation_lag_7_day_h',

                'ratio_lag_14_7_day',
            ]

    
    elif is_consumption==1:
        fts_to_add += [
            # 'euros_per_mwh',
            # 'euros_per_mwh_lag_7_day_mean_diff',

            
#             'euros_per_mwh_lag_28_day_mean_diff',
#             'euros_per_mwh_lag_7_day_diff',
#             'euros_per_mwh_lag_2_day_diff',
            "lag_8_14_day_mean",
#             'daily_euros_per_mwh',
#             "lag_7_day_1h",
#             'direct_solar_radiation_7_day_mean_diff', 
#             'cloudcover_low_7_day_mean_diff',
#             'dewpoint_7_day_mean_diff',
#             'total_precipitation_7_day_mean_diff',
#             'surface_solar_radiation_downwards_7_day_mean_diff',
#             "daytype_lag_2_day",
#             'direct_solar_radiation_7_day_diff', 
#             "daytype_lag_7_day",
#             'lag_2_day_1h_consumption_counterpart',
#             "rolling_mean_24x7x4",
#             "school_holiday",
#                 'eic_count_7_day_mean_diff',
#                 'installed_capacity_7_day_mean_diff',

            'direct_solar_radiation_7_day_mean_diff', 
            
            # 'temp_above_cp',
            # 'temp_below_cp',

            
        ]

        if target in ['target']:
            fts_to_add += [
                'temperature_lag_1h',
                'temperature_7_day_diff', 
                'temperature_8_14_day_mean_diff',
                'temperature_7_day_diff',


                'temperature_7_day_mean_diff',
                'rolling_mean_24_lag_7_day',

                "ratio_lag_21_7_day",
            ]

            fts_to_remove += [
                # 'target_all_type_lag_2_day',
                # 'target_all_type_lag_7_day',
                
            ]
            
        if target in ['lag_7_day_mean_diff', 'lag_7_day_mean_diff_neg']:
            fts_to_add += [
                'temperature_7_day_mean_diff',
                'rolling_mean_24_lag_7_day',

                "ratio_lag_14_7_day",


                "lag_5_day",
                "lag_6_day",
            ]
            
        if target in ['lag_7_day_1h_mean_diff', 'lag_7_day_1h_mean_diff_neg']:
            fts_to_add += [
                'temperature_7_day_1h_mean_diff',
                'rolling_mean_24_lag_7_day',

                "ratio_lag_14_7_day",


                "lag_5_day",
                "lag_6_day",
            ]
        
        if target in ['lag_2_day_diff', 'lag_2_day_diff_neg']:
            fts_to_add += [
                # "lag_2_day_1h_consumption_counterpart",
                # 'dayofweek_lag_2_day',

                'ratio_lag_9_7_day',
                # 'ratio_lag_16_14_day',

                'ratio_rolling_mean_lag_9_7_day',
                
            ]


        if target in ['lag_7_day_diff', 'lag_7_day_diff_neg']:
            fts_to_add += [
                'direct_solar_radiation_7_day_1h_diff', 
                'temperature_7_day_1h_diff', 

                "ratio_lag_14_7_day",

                "ratio_lag_21_7_day",

                'temperature_14_7_day_diff',
            ]
    else:
        raise ValueError

        
    if target in ['lag_2_day_diff', 'lag_2_day_diff_neg']:
        fts_to_add += [
            'daytype_lag_2_day',
            
            'temperature_2_day_diff', 
            'direct_solar_radiation_2_day_diff', 
            'surface_solar_radiation_downwards_2_day_diff',
            'total_precipitation_2_day_diff', 
            
            'cloudcover_total_2_day_diff', 
            
            
            # 'euros_per_mwh_lag_2_day_diff',
#             'eic_count_2_day_diff',
#             'installed_capacity_2_day_diff',
        ]


        fts_to_remove += [
            # "euros_per_mwh_lag_7_day_mean_diff",


                'eic_count_7_day_mean_diff',
                'installed_capacity_7_day_mean_diff',
        ]


    if target in ['lag_3_day_diff', 'lag_3_day_diff_neg']:
        fts_to_add += [
            'daytype_lag_3_day',
            
            'temperature_3_day_diff', 
            'direct_solar_radiation_3_day_diff', 
            'surface_solar_radiation_downwards_3_day_diff',
            'total_precipitation_3_day_diff', 
            
            'cloudcover_total_3_day_diff', 

            'direct_solar_radiation_3_day_1h_diff',
            'temperature_3_day_1h_diff',
            "lag_3_day_1h",
            # 'euros_per_mwh_lag_2_day_diff',
#             'eic_count_2_day_diff',
#             'installed_capacity_2_day_diff',
        ]


        fts_to_remove += [
            # "euros_per_mwh_lag_7_day_mean_diff",


                'eic_count_7_day_mean_diff',
                'installed_capacity_7_day_mean_diff',
        ]

    elif target in ['lag_7_day_1h_mean_diff', 'lag_7_day_1h_mean_diff_neg']:
        fts_to_add += [
            'direct_solar_radiation_7_day_mean_diff',
            'surface_solar_radiation_downwards_7_day_mean_diff',
            'lag_7_day_1h_mean',
        ]
        
    elif target in ['lag_7_day_diff', 'lag_7_day_diff_neg']:
        fts_to_add += [
            'daytype_lag_7_day',
            
            "lag_7_day_consumption_counterpart",
            'temperature_7_day_diff', 
            'direct_solar_radiation_7_day_diff', 
            'surface_solar_radiation_downwards_7_day_diff',
            'total_precipitation_7_day_diff', 
            
            'cloudcover_total_7_day_diff', 
            
            
            # 'euros_per_mwh_lag_7_day_diff',
            'eic_count_7_day_diff',
            'installed_capacity_7_day_diff',


            
            "lag_7_day_1h",
            "lag_7_day_m1h",
        ]
        
        fts_to_remove += [
            # "euros_per_mwh_lag_7_day_mean_diff",
            # "daytype_lag_3_day",
            'temperature_2_day_1h_diff',
        ]
        
    elif target in ['lag_14_day_diff', 'lag_14_day_diff_neg']:
        fts_to_add += [

            # 'daytype_lag_7_day',
            
            "lag_14_day_consumption_counterpart",
            'temperature_14_day_diff', 
            'direct_solar_radiation_14_day_diff', 
            'surface_solar_radiation_downwards_14_day_diff',
            'total_precipitation_14_day_diff', 
            
            # 'cloudcover_total_14_day_diff', 
            
            
            # 'euros_per_mwh_lag_7_day_diff',
            # 'eic_count_7_day_diff',
            # 'installed_capacity_7_day_diff',

            "lag_14_day",
            # "lag_7_day_m1h",


            # 'eic_count_14_day_diff', 
            # 'installed_capacity_14_day_diff',
        ]
        
        fts_to_remove += [
            # "euros_per_mwh_lag_7_day_mean_diff",
            # "daytype_lag_3_day",
            'temperature_2_day_1h_diff',
        ]

    
    features += fts_to_add
        
        
    for ft in fts_to_remove:
        try:
            features.remove(ft)
        except:
            if verbose:
                print(f'Removing aborted: {ft} not found')
        
    if verbose==1:
        print(f'removing features: {fts_to_remove}')
        print(f'Adding features: {fts_to_add}')
        print(f'Final features: ({len(features)}) {features}')
        
    return list(set(features))

In [29]:
import re

In [30]:
def get_monotone_constraints_map(features, is_consumption, is_business=0):
    
    monotone_constraints_map = {ft: 0 for ft in features}
    

    if is_consumption==0:
#         monotone_constraints_map['euros_per_mwh'] = 1
#         monotone_constraints_map['daily_euros_per_mwh'] = 1
#         monotone_constraints_map['wind_speed'] = 1
    
        monotone_constraints_map['installed_capacity'] = 1
        
        monotone_constraints_map["direct_solar_radiation"] = 1
        monotone_constraints_map["cloudcover_total"] = -1
        monotone_constraints_map["cloudcover_mid"] = -1
        monotone_constraints_map["cloudcover_low"] = -1
        
        
#         monotone_constraints_map['cloudcover_low_2_day_diff'] = -1
#         monotone_constraints_map['direct_solar_radiation_2_day_diff'] = 1
#         monotone_constraints_map['surface_solar_radiation_downwards_2_day_diff'] = 1

#         monotone_constraints_map['cloudcover_low_7_day_mean_diff'] = -1
        
        # monotone_constraints_map['direct_solar_radiation_7_day_mean_diff'] = 1
        # monotone_constraints_map['surface_solar_radiation_downwards_7_day_mean_diff'] = 1

        # monotone_constraints_map['direct_solar_radiation'] = 1
        # monotone_constraints_map['surface_solar_radiation_downwards'] = 1


        # monotone_constraints_map['snowfall_1_day_rolling_sum'] = -1

        # for ft in features:
        #     if re.search('direct_solar_radiation_[0-9]_day_diff', ft):
        #         monotone_constraints_map[ft] = 1
        #     if re.search('surface_solar_radiation_downwards_[0-9]_day_diff', ft):
        #         monotone_constraints_map[ft] = 1
#             if re.search('euros_per_mwh_lag_[0-9]_day_diff', ft):
#                 monotone_constraints_map[ft] = 1
                
    elif is_consumption==1:
        # pass
#         monotone_constraints_map['euros_per_mwh'] = -1
#         monotone_constraints_map['daily_euros_per_mwh'] = -1
        monotone_constraints_map['eic_count'] = 1
    
    else:
        raise valueError

#         monotone_constraints_map['direct_solar_radiation_7_day_mean_diff'] = 1



        
    return monotone_constraints_map

In [31]:
def append_diffs(df):


    df['lag_7_day_mean_diff'] = np.clip(df['target'] - df['lag_7_day_mean'], 0, np.inf)
    df['lag_7_day_mean_diff_neg'] = np.clip(df['lag_7_day_mean'] - df['target'], 0, np.inf)

    df['lag_7_day_1h_mean_diff'] = np.clip(df['target'] - df['lag_7_day_1h_mean'], 0, np.inf)
    df['lag_7_day_1h_mean_diff_neg'] = np.clip(df['lag_7_day_1h_mean'] - df['target'], 0, np.inf)

    df['lag_2_day_diff'] = np.clip(df['target'] - df['lag_2_day'], 0, np.inf)
    df['lag_2_day_diff_neg'] = np.clip(df['lag_2_day'] - df['target'], 0, np.inf)
    
    df['lag_3_day_diff'] = np.clip(df['target'] - df['lag_3_day'], 0, np.inf)
    df['lag_3_day_diff_neg'] = np.clip(df['lag_3_day'] - df['target'], 0, np.inf)
    
    df['lag_7_day_diff'] = np.clip(df['target'] - df['lag_7_day'], 0, np.inf)
    df['lag_7_day_diff_neg'] = np.clip(df['lag_7_day'] - df['target'], 0, np.inf)
    
    df['lag_14_day_diff'] = np.clip(train['target'] - df['lag_14_day'], 0, np.inf)
    df['lag_14_day_diff_neg'] = np.clip(train['lag_14_day'] - df['target'], 0, np.inf)
    
    for target in [
            'target', 
           'lag_7_day_mean_diff', 
           'lag_7_day_mean_diff_neg',
           'lag_7_day_1h_mean_diff', 
           'lag_7_day_1h_mean_diff_neg',
           'lag_2_day_diff', 
           'lag_2_day_diff_neg',
           'lag_3_day_diff', 
           'lag_3_day_diff_neg',
           'lag_7_day_diff',
           'lag_7_day_diff_neg',
#            'lag_14_day_diff',
#            'lag_14_day_diff_neg',
          ]:
        
        df['scaled_'+target] = df[target] / (df['installed_capacity'].fillna(0) +1000) * 1000
        df.loc[df['is_consumption']==1, 'scaled_'+target] = df.loc[train['is_consumption']==1, target] / (df.loc[df['is_consumption']==1, 'eic_count'].fillna(0)+100) * 100

    
    return df

In [32]:
train = append_diffs(train)

In [33]:
np.random.seed(42)

# train_augmented = ts_cv_train.sort_values(['datetime'])[-400000:].copy()
train_augmented = train.sort_values(['datetime'])[-400000:].copy()

aug_factor = np.random.rand(400000) * 0.6 + 1 # np.random.rand(200000) * 0.05 + 0.3 + 1

for target in [

    
        'target', 
       'lag_7_day_mean_diff', 
       'lag_7_day_mean_diff_neg',

       'lag_7_day_1h_mean_diff', 
       'lag_7_day_1h_mean_diff_neg',
    
       'lag_2_day_diff', 
       'lag_2_day_diff_neg',
       'lag_3_day_diff', 
       'lag_3_day_diff_neg',
       'lag_7_day_diff',
       'lag_7_day_diff_neg',

       # 'lag_14_day_diff',
       # 'lag_14_day_diff_neg',
    
        'lag_2_day_consumption_counterpart', 
        "lag_2_day_1h_consumption_counterpart",
        "lag_3_day_consumption_counterpart",
        'lag_7_day_consumption_counterpart', 
        'lag_7_day_mean_consumption_counterpart', 
        # 'rolling_mean_24_consumption_counterpart',


        'installed_capacity',
        'eic_count',
        'eic_count_7_day_mean_diff', 
        'installed_capacity_7_day_mean_diff',

        "lag_2_day",
        "lag_2_day_1h",
        "lag_3_day",
        "lag_3_day_1h",
        "lag_4_day",
        "lag_7_day_mean",
        "lag_5_day",
        "lag_6_day",
        "lag_7_day",
        "lag_7_day_1h",
        "lag_7_day_m1h",
        "lag_14_day",
        
        "rolling_mean_24",
        # "rolling_mean_24x7",
        "rolling_std_24",
        "lag_8_14_day_mean",
    
        # 'lag_7_day_1h_mean',
        # "target_all_type",
        # 'target_all_type_lag_2_day',
        # 'target_all_type_lag_7_day',
        # 'consumption_counterpart_all_type_lag_2_day',
        # 'consumption_counterpart_all_type_lag_7_day',

        'target_all_lag_2_day',
        'target_all_lag_7_day',
        'consumption_counterpart_all_lag_2_day',
        'consumption_counterpart_all_lag_3_day',
        'consumption_counterpart_all_lag_7_day',



        # 'ratio_lag_9_2_day',
        # 'ratio_lag_14_7_day',
        # 'ratio_lag_9_7_day',
      ]:
    train_augmented[target] = train_augmented[target] * aug_factor

In [34]:
import gc
try:
    for i in range(15, 22):
        del train[f'lag_{i}_day'] 
except:
    pass
gc.collect()

0

## Train Models

In [35]:
train_augmented = pd.concat([train, train_augmented])

In [36]:
import sklearn

In [37]:
pd.__version__, np.__version__, lgb.__version__, sklearn.__version__

('2.0.3', '1.24.3', '4.2.0', '1.2.2')

In [38]:
%%time

k_fold = 6
np.random.seed(42)
gkf = GroupKFold(n_splits=k_fold, )

CPU times: user 33 µs, sys: 8 µs, total: 41 µs
Wall time: 43.6 µs


In [39]:
import pickle

In [40]:
models_path = "/kaggle/input/enefit-trained-gbts/0127e_custom_3fcv"

results = {}

for is_consumption in [0, 1]:

    
    targets = [
        'target', 
       'lag_7_day_mean_diff', 
       'lag_7_day_mean_diff_neg',
       'lag_7_day_1h_mean_diff', 
       'lag_7_day_1h_mean_diff_neg',
       'lag_2_day_diff', 
       'lag_2_day_diff_neg',
       'lag_3_day_diff', 
       'lag_3_day_diff_neg',
       'lag_7_day_diff',
       'lag_7_day_diff_neg',
#         'lag_14_day_diff',
#        'lag_14_day_diff_neg',
      ]
    


    for target_prefix in ["", "scaled_"]:
#     for target_prefix in ["scaled_"]:
        for target in targets:
            key = (is_consumption, target_prefix + target)
            print(key)

            with open(f"{models_path}/{key[0]}_{key[1]}.cvbooster", "rb") as file:
                results[key] = pickle.load(file)

(0, 'target')
(0, 'lag_7_day_mean_diff')
(0, 'lag_7_day_mean_diff_neg')
(0, 'lag_7_day_1h_mean_diff')
(0, 'lag_7_day_1h_mean_diff_neg')
(0, 'lag_2_day_diff')
(0, 'lag_2_day_diff_neg')
(0, 'lag_3_day_diff')
(0, 'lag_3_day_diff_neg')
(0, 'lag_7_day_diff')
(0, 'lag_7_day_diff_neg')
(0, 'scaled_target')
(0, 'scaled_lag_7_day_mean_diff')
(0, 'scaled_lag_7_day_mean_diff_neg')
(0, 'scaled_lag_7_day_1h_mean_diff')
(0, 'scaled_lag_7_day_1h_mean_diff_neg')
(0, 'scaled_lag_2_day_diff')
(0, 'scaled_lag_2_day_diff_neg')
(0, 'scaled_lag_3_day_diff')
(0, 'scaled_lag_3_day_diff_neg')
(0, 'scaled_lag_7_day_diff')
(0, 'scaled_lag_7_day_diff_neg')
(1, 'target')
(1, 'lag_7_day_mean_diff')
(1, 'lag_7_day_mean_diff_neg')
(1, 'lag_7_day_1h_mean_diff')
(1, 'lag_7_day_1h_mean_diff_neg')
(1, 'lag_2_day_diff')
(1, 'lag_2_day_diff_neg')
(1, 'lag_3_day_diff')
(1, 'lag_3_day_diff_neg')
(1, 'lag_7_day_diff')
(1, 'lag_7_day_diff_neg')
(1, 'scaled_target')
(1, 'scaled_lag_7_day_mean_diff')
(1, 'scaled_lag_7_day_mean_d

In [41]:
if lgb.__version__ == '4.2.0':
    stat_prefix = "valid "
else:
    stat_prefix = ""

try:
    display(
        pd.DataFrame(
         [(is_consumption, 
           target, 
           results[(is_consumption, target)][f'{stat_prefix}l1-mean'][-1],
           results[(is_consumption, target)][f'{stat_prefix}l1-stdv'][-1],
           results[(is_consumption, target)]['cvbooster'].boosters[0].num_trees(),
          ) for is_consumption, target in list(results.keys())], 
            columns=['is_consumption', 'target', 'mean', 'std', 'num_trees'],
     )
    )
except Exception as e:
    print(e)
    

,is_consumption,target,mean,std,num_trees
0,0,target,35.755446,2.499104,598
1,0,lag_7_day_mean_diff,17.639779,3.003372,795
2,0,lag_7_day_mean_diff_neg,19.488621,1.890421,585
3,0,lag_7_day_1h_mean_diff,18.354310,2.554906,638
4,0,lag_7_day_1h_mean_diff_neg,19.628460,1.712101,460
5,0,lag_2_day_diff,19.597315,1.914070,905
6,0,lag_2_day_diff_neg,19.067033,1.309939,632
7,0,lag_3_day_diff,19.488663,3.099069,839
8,0,lag_3_day_diff_neg,20.978250,0.830244,431
9,0,lag_7_day_diff,19.070944,3.433834,883


In [42]:
def get_boosters(key, results, i=None):


    if isinstance(results[key], lgb.basic.Booster):
        return [results[key]]

    
    boosters = results[key]["cvbooster"].boosters
    
    if i is not None:
        boosters = [boosters[i]]
        
    return boosters

def get_prediciton(input_df_ori, results, is_consumptions=[0, 1], i=None, scaled=False, epsilon=100):
    
    input_df = input_df_ori.copy()
    try:
        del input_df['pred']
    except:
        pass
    
    input_df['pred'] = None
    
    keys = list(results.keys())
    
    preds={
        0: [],
        1: [],
    } 

    for is_consumption in is_consumptions:
        
        if is_consumption:
            # coef_0, coef_1, coef_2, coef_3 = 1/5, 1/5, 1/5, 2/5
            # coef_0, coef_1, coef_2, coef_3 = 2/9, 2/9, 2/9, 3/9
            # coef_0, coef_1, coef_2, coef_3 = 1/4, 1/4, 1/4, 1/4
            # coefs = [2/14, 2/14, 2/14, 1/14, 2/14, 2/14, 2/14, 1/14]
            # coefs = [2/16, 2/16, 2/16, 1/16, 1/16, 2/16, 2/16, 2/16, 1/16, 1/16]
#             coefs = [2/20, 2/20, 2/20, 2/20, 1/20, 1/20, 2/20, 2/20, 2/20, 2/20, 1/20, 1/20]
#             coefs = [2/16, 2/16, 0/16, 2/16, 1/16, 1/16, 2/16, 2/16, 0/16, 2/16, 1/16, 1/16]
            coefs = [2/18, 2/18, 1/18, 2/18, 1/18, 1/18, 2/18, 2/18, 1/18, 2/18, 1/18, 1/18]

        else:
            # coef_0, coef_1, coef_2, coef_3 = 2/9, 2/9, 2/9, 3/9
             # coef_0, coef_1, coef_2, coef_3 = 1/4, 1/4, 1/4, 1/4
#             coefs = [0.06451613, 0.16129032, 0.12903226, 0.12903226, 
#                      0.25806452, 0.16129032, 0.06451613, 0.03225806]
            # coefs = [2/14, 2/14, 2/14, 1/14, 2/14, 2/14, 2/14, 1/14]
            # coefs = [2/16, 2/16, 2/16, 1/16, 1/16, 2/16, 2/16, 2/16, 1/16, 1/16]
#             coefs = [2/20, 2/20, 2/20, 2/20, 1/20, 1/20, 2/20, 2/20, 2/20, 2/20, 1/20, 1/20]
#             coefs = [2/16, 2/16, 0/16, 2/16, 1/16, 1/16, 2/16, 2/16, 0/16, 2/16, 1/16, 1/16]
            coefs = [2/18, 2/18, 1/18, 2/18, 1/18, 1/18, 2/18, 2/18, 1/18, 2/18, 1/18, 1/18]
    

        input_df_subset = input_df[
            (input_df['is_consumption'] == is_consumption)]
        
#         print(len(input_df_subset))

        n_models = 22


        
        # if scaled:
        scale_factor = [
            (input_df_subset['installed_capacity'].fillna(0) + (10*epsilon)) / (10*epsilon),
            (input_df_subset['eic_count'].fillna(0) + epsilon) / epsilon,
                           ]
        # # else:
        non_scale_factor = [1, 1]



        
        j = 0
        while j < n_models:
            key = keys[is_consumption*n_models + j]


            if key[1][:6] == "scaled":
                scale = scale_factor[is_consumption]
            elif key[1][:6] == "new_fe":
                scale = scale_factor[is_consumption] * (input_df_subset['temperature']+ 273.15) / 200
            else:
                scale = non_scale_factor[is_consumption]
            
            if key[1][-6:] == 'target':
                boosters = get_boosters(keys[is_consumption*n_models + j], results, i)
                pred = np.array([model.predict(input_df_subset[boosters[0].feature_name()]) for model in boosters]).mean(axis=0) * scale
                j += 1
            elif key[1][-19:] == "lag_7_day_mean_diff":
                boosters = get_boosters(keys[is_consumption*n_models + j], results, i)
                pred_pos = np.array([model_pos.predict(input_df_subset[boosters[0].feature_name()]) for model_pos in boosters]).mean(axis=0)
                boosters = get_boosters(keys[is_consumption*n_models + j+1], results, i)
                pred_neg = np.array([model_neg.predict(input_df_subset[boosters[0].feature_name()]) for model_neg in boosters]).mean(axis=0)
                pred = (pred_pos - pred_neg) * scale + input_df_subset['lag_7_day_mean']
                j+=2
            elif key[1][-22:] == "lag_7_day_1h_mean_diff":
                boosters = get_boosters(keys[is_consumption*n_models + j], results, i)
                pred_pos = np.array([model_pos.predict(input_df_subset[boosters[0].feature_name()]) for model_pos in boosters]).mean(axis=0)
                boosters = get_boosters(keys[is_consumption*n_models + j+1], results, i)
                pred_neg = np.array([model_neg.predict(input_df_subset[boosters[0].feature_name()]) for model_neg in boosters]).mean(axis=0)
                pred = (pred_pos - pred_neg) * scale + input_df_subset['lag_7_day_1h_mean']
                j+=2
            elif key[1][-14:] == "lag_2_day_diff":
                boosters = get_boosters(keys[is_consumption*n_models + j], results, i)
                pred_pos = np.array([model_pos.predict(input_df_subset[boosters[0].feature_name()]) for model_pos in boosters]).mean(axis=0)
                boosters = get_boosters(keys[is_consumption*n_models + j+1], results, i)
                pred_neg = np.array([model_neg.predict(input_df_subset[boosters[0].feature_name()]) for model_neg in boosters]).mean(axis=0)
                pred = (pred_pos - pred_neg) * scale + input_df_subset['lag_2_day']
                # pred = ((pred_pos - pred_neg) / 100 + 1 )* input_df_subset['lag_2_day']
                j+=2
            elif key[1][-14:] == "lag_3_day_diff":
                boosters = get_boosters(keys[is_consumption*n_models + j], results, i)
                pred_pos = np.array([model_pos.predict(input_df_subset[boosters[0].feature_name()]) for model_pos in boosters]).mean(axis=0)
                boosters = get_boosters(keys[is_consumption*n_models + j+1], results, i)
                pred_neg = np.array([model_neg.predict(input_df_subset[boosters[0].feature_name()]) for model_neg in boosters]).mean(axis=0)
                pred = (pred_pos - pred_neg) * scale + input_df_subset['lag_3_day']
                j+=2
            elif key[1][-14:] == "lag_7_day_diff":
                boosters = get_boosters(keys[is_consumption*n_models + j], results, i)
                pred_pos = np.array([model_pos.predict(input_df_subset[boosters[0].feature_name()]) for model_pos in boosters]).mean(axis=0)
                boosters = get_boosters(keys[is_consumption*n_models + j+1], results, i)
                pred_neg = np.array([model_neg.predict(input_df_subset[boosters[0].feature_name()]) for model_neg in boosters]).mean(axis=0)
                pred = (pred_pos - pred_neg) * scale + input_df_subset['lag_7_day']
                j+=2
            elif key[1][-15:] == "lag_14_day_diff":
                boosters = get_boosters(keys[is_consumption*n_models + j], results, i)
                pred_pos = np.array([model_pos.predict(input_df_subset[boosters[0].feature_name()]) for model_pos in boosters]).mean(axis=0)
                boosters = get_boosters(keys[is_consumption*n_models + j+1], results, i)
                pred_neg = np.array([model_neg.predict(input_df_subset[boosters[0].feature_name()]) for model_neg in boosters]).mean(axis=0)
                pred = (pred_pos - pred_neg) * scale + input_df_subset['lag_14_day']
                j+=2
            else:
                raise ValueError
                

                
            preds[is_consumption].append(pred)

        if n_models == 7:
            pred = np.array(preds[is_consumption][0]*coefs[0] + preds[is_consumption][1]*coefs[1] + preds[is_consumption][2]*coefs[2] + preds[is_consumption][3]*coefs[3]).clip(min=0)
            
        elif n_models == 14:
            pred = np.array(
                preds[is_consumption][0]*coefs[0] + preds[is_consumption][1]*coefs[1] + preds[is_consumption][2]*coefs[2] + preds[is_consumption][3]*coefs[3] +
                preds[is_consumption][4+0]*coefs[4+0] + preds[is_consumption][4+1]*coefs[4+1] + preds[is_consumption][4+2]*coefs[4+2] + preds[is_consumption][4+3]*coefs[4+3]
                ).clip(min=0)
            
        elif n_models == 18:
            pred = np.array(
                preds[is_consumption][0]*coefs[0] + preds[is_consumption][1]*coefs[1] + preds[is_consumption][2]*coefs[2] + preds[is_consumption][3]*coefs[3] + preds[is_consumption][4]*coefs[4] +
                preds[is_consumption][5+0]*coefs[5+0] + preds[is_consumption][5+1]*coefs[5+1] + preds[is_consumption][5+2]*coefs[5+2] + preds[is_consumption][5+3]*coefs[5+3] + preds[is_consumption][5+4]*coefs[5+4]
                ).clip(min=0)
            
        elif n_models == 22:
            pred = np.array(
                sum([preds[is_consumption][i]*coefs[i] for i in range(12)])
                # preds[is_consumption][0]*coefs[0] + preds[is_consumption][1]*coefs[1] + preds[is_consumption][2]*coefs[2] + preds[is_consumption][3]*coefs[3] + preds[is_consumption][4]*coefs[4] +
                # preds[is_consumption][5+0]*coefs[5+0] + preds[is_consumption][5+1]*coefs[5+1] + preds[is_consumption][5+2]*coefs[5+2] + preds[is_consumption][5+3]*coefs[5+3] + preds[is_consumption][5+4]*coefs[5+4]
                ).clip(min=0)
            
        else:
            raise ValueError
        

        
        input_df.loc[
            (input_df['is_consumption'] == is_consumption),
#                 * (test['is_business'] == is_business), 
            'pred'] = pred
        
    # return [pred_0, pred_1, pred_2, pred_3, input_df]
    return [preds, input_df]
    # return [preds[0], input_df]

In [43]:
def continue_training(results, new_data):

    tick = time.time()
    
    for is_consumption in [0, 1]:

        train_subset = new_data[
            (new_data['is_consumption'] == is_consumption)
        ]
        
        for target_prefix in ["", "scaled_"]:
            
            for target in [
                'target', 
               'lag_7_day_mean_diff', 
               'lag_7_day_mean_diff_neg',
               'lag_2_day_diff', 
               'lag_2_day_diff_neg',
               'lag_7_day_diff',
               'lag_7_day_diff_neg'
              ]:
                
                full_target_name = target_prefix + target
                
                key = (is_consumption, full_target_name)

#                 boosters = results[key]['cvbooster'].boosters
                boosters = [results[key]]
                
                features = boosters[0].feature_name()
                train_subset.reset_index(drop=True, inplace=True)

    #             print(key, features)

                X = train_subset[features]
                Y = train_subset[full_target_name]

#                 G = train_subset['day']

#                 folds = gkf.split(X, Y, G)

                lgb_train = lgb.Dataset(X, 
                                        Y, 
                                        free_raw_data=False,
                                       )

                params = {
                    'boosting_type': 'gbdt',
                    'objective': 'tweedie',
        #             'objective': 'l1',
                    # 'tree_learner': 'feature',
                    # 'max_bin': 63,
                    'metric': 'mae', 
                    'learning_rate': 0.1 / 5,
                    'feature_fraction': 0.9 - 0.2 + 0.2*(target=='target'), 
        #             'feature_fraction': 0.9 - 0.2, 
                    'cat_l2': 0.1 + 1 * is_consumption,
                    'reg_sqrt': True,
                    'max_depth': 7,
                    'num_leaves': 127,
                    'monotone_constraints': [value*((key[-3:]!='neg')*2-1) for key, value in get_monotone_constraints_map(features, is_consumption).items()],
                    'monotone_constraints_method': 'advanced',
        #             'weight_column': 'weight',
                    'random_state': 42,
                    # 'monotone_penalty': 0.01,
        #             'bagging_fraction': 0.8,
        #             'bagging_freq': 2,
                    'reg_alpha': 0.2 + 1 * is_consumption, 
                    'reg_lambda': 0.2 + 1 * is_consumption, 
                #     'extra_trees': True,
                    'verbose': -1,
                    # 'device_type': 'cuda' 
                }

                if is_consumption==0:
                    params['tweedie_variance_power'] = 1.4
                elif is_consumption==1:
                    params['tweedie_variance_power'] = 1.25

                categories = ['county','product_type', 'daytype', 'dayofweek']
                if is_consumption:
                    categories += ['hour']

                num_boost_round = 5
#                 for i in range(k_fold):
                for i in range(1):

                    boosters[i] = lgb.train(
                        params=params,
                        train_set = lgb_train,
            #             lgb_train,
                        categorical_feature=categories, 
            #             folds=folds,
                        num_boost_round=num_boost_round,
                        init_model=boosters[i],
                        keep_training_booster =True,
            #             num_boost_round=50,
            #             callbacks=[
            #                 lgb.early_stopping(stopping_rounds=100, verbose=True),
            #                 lgb.log_evaluation(50)
            #             ],
            #             return_cvbooster=True
                    )




    #             results[key] = lgb.cv(
    #                 params=params,
    #                 train_set = lgb_train,
    #                 categorical_feature=categories, 
    #                 folds=folds,
    #                 num_boost_round=num_boost_round,
    #                 init_model=boosters,
    #     #             num_boost_round=50,
    #     #             callbacks=[
    #     #                 lgb.early_stopping(stopping_rounds=100, verbose=True),
    #     #                 lgb.log_evaluation(50)
    #     #             ],
    #                 return_cvbooster=True
    #             )
    
    print(f"Finished training with {len(new_data)} new rows of data for {num_boost_round} rounds in {time.time() -tick:.1f}s")
    
    return results

In [44]:
# # %%time

# MAE_by_shift = [
#     [[], [], [], [], [], [], [], [], [], [], [], [],],
#     [[], [], [], [], [], [], [], [], [], [], [], [],]
# ]

# for is_consumption in [0, 1]:
# # for is_consumption in [1]:
    
#     gc.collect()

#     train_subset = train[
#         (train['is_consumption'] == is_consumption)
#     #             * (train['is_business'] == is_business)
#     ]


#     train_subset = train_subset.reset_index(drop=True)[:]



#     X = train_subset#[features]
#     Y = train_subset['target']

# #     G = train_subset['month']
# #     G = train_subset['day']
#     G = train_subset['hour']


#     error_dfs = []
#     MAEs = []

    

#     # for i, (train_index, test_index) in enumerate(gkf.split(X, Y, G)):
#     for i, (train_index, test_index) in enumerate(folds):
#         index = test_index

#         preds = get_prediciton(train_subset.iloc[index], results, is_consumptions=[is_consumption], i=i, scaled=True, epsilon=100)
# #         preds_2 = get_prediciton(train_subset.iloc[index], results_2, is_consumptions=[is_consumption], i=i, scaled=True, epsilon=10)
# #         preds[-1] = preds[-1]['pred']*0.5 + preds_2[-1]['pred']*0.5

#         preds[-1] = preds[-1]['pred']
#         targets = train_subset.iloc[index]['target']


#     #     pred = preds.mean(axis=0)

#         print(f"MAE_SCORE:", end='')
#         for j, pred in enumerate(preds[0][is_consumption]):

#             mae = abs(targets - pred)
#             MAE_by_shift[is_consumption][j].append(mae.mean())
#             print(f"\t{mae.mean():.2f}", end='')

#         print("", end='')
#         mae = abs(targets - preds[-1])
#         # MAE_by_shift[is_consumption][j].append(mae.mean())
#         print(f"\t{mae.mean():.2f}", end='')

#         print('')

#         MAEs.append(mae.mean())

#         # error_df = train_subset.iloc[index][features+['month']].copy()
#         error_df = train_subset.iloc[index].copy()
#         error_df['MAE'] = mae
#         error_df['ME'] = pred - targets

#         error_dfs.append(error_df)

#     print(f'mean mea: {np.mean(MAEs):.2f}')
#     agg_error_df = pd.concat(error_dfs)



In [45]:
# MAE_SCORE:	15.59	15.09	14.34	15.91	16.07	15.48	13.54	13.81	13.61
# MAE_SCORE:	14.99	14.43	13.85	15.44	15.61	14.93	13.01	13.19	13.11
# MAE_SCORE:	15.77	15.23	14.77	16.23	16.14	15.68	13.87	13.97	13.83
# MAE_SCORE:	17.19	16.36	16.15	17.65	17.48	16.82	15.16	15.23	14.98
# MAE_SCORE:	17.42	16.65	16.18	17.58	17.63	17.00	15.32	15.42	15.06
# MAE_SCORE:	16.68	16.02	15.42	17.08	17.01	16.37	14.49	14.89	14.46
# mean mea: 14.17
# MAE_SCORE:	27.28	28.90	31.45	28.15	30.08	29.12	31.04	25.30	26.05

In [46]:
# np.mean(MAE_by_shift, axis=2)

In [47]:
   
# for ft in ['day', 'county', 'product_type', 'daytype', 'dayofweek', 'month']:
# # for ft in ['county', 'is_consumption', 'is_business', 'product_type',]:
#     plt.figure(figsize=(9,4))
#     sns.boxenplot(
#         agg_error_df[[ft, 'ME']], x=ft, y='ME', outlier_prop=0.03,showfliers=False, k_depth='proportion',
#     )
#     plt.show()

#     print(f"feature: {ft}")
#     for feature_value in agg_error_df[ft].unique():
#         me = agg_error_df[agg_error_df[ft]==feature_value]['ME'].mean()
#         mae = abs(agg_error_df[agg_error_df[ft]==feature_value]['ME']).mean()
#         print(f"\tfeature_value: {feature_value}, ME {me:.2f}, MAE {mae:.2f}")
            


In [48]:
# # import lightgbm

# for key in results.keys():
#     print(key)
#     importance_split = np.mean([booster.feature_importance(importance_type='split') / booster.feature_importance(importance_type='split').sum()
#                      for booster in results[key]['cvbooster'].boosters], axis=0) * 100
    
#     importance_gain = np.mean([booster.feature_importance(importance_type='gain') / booster.feature_importance(importance_type='gain').sum()
#                      for booster in results[key]['cvbooster'].boosters], axis=0) * 100
    
#     importance = importance_gain / importance_split
    
#     feature_name = results[key]['cvbooster'].boosters[0].feature_name()
    
# #     for imp in [importance_split, importance_gain, importance]:
#     for imp in [importance_gain]:
#         tuples = sorted(zip(feature_name, imp), key=lambda x: x[1], reverse=True)

#         plt.figure(figsize=(10,8))
#         ax = sns.barplot(pd.DataFrame(tuples), x=1, y=0, orient='h')
#         for i in ax.containers:
#             ax.bar_label(i,)
#         plt.show()

In [49]:
# agg_error_df['day'] = agg_error_df['datetime'].dt.day

In [50]:
# sns.boxenplot(
#     agg_error_df[['day', 'MAE']], x='day', y='MAE', outlier_prop=0.03,showfliers=False, k_depth='proportion',
# )
# plt.show()

In [51]:
# sns.boxenplot(
#     agg_error_df[['hour', 'ME']], x='hour', y='ME', outlier_prop=0.03,showfliers=False, k_depth='proportion',
# )
# plt.show()

In [52]:
# plt.figure(figsize=(14,8))
# sns.scatterplot(
#     agg_error_df[['date', 'ME']].groupby('date').mean().reset_index(), x='date', y='ME', marker='.')

## Make pred & Submission¶

In [53]:
# inspect = []


def append_lag(test, placeholder_previous_day, historic_train, predicting_start_time, client_df=None):
#     inspect.append(test)
    
    new_df = pd.concat([test, placeholder_previous_day, historic_train,])
    
    new_df, _ = fill_missing_time(new_df, start_time=placeholder_previous_day['datetime'].min()-datetime.timedelta(days=30), inference=True)
    
    
    new_df = append_daytype(new_df)
#     new_df = append_school_holiday(new_df)

    del new_df['eic_count']
    del new_df['installed_capacity']
    del new_df['available_date']
    # del new_df['target_all_type']
    # del new_df['target_all_type_lag_2_day']
    # del new_df['target_all_type_lag_7_day']
    # del new_df['consumption_counterpart_all_type_lag_2_day']
    # del new_df['consumption_counterpart_all_type_lag_7_day']
    del new_df['target_all_lag_2_day']
#     del new_df['target_all_lag_3_day']
    del new_df['target_all_lag_7_day']
    del new_df['consumption_counterpart_all_lag_2_day']
    del new_df['consumption_counterpart_all_lag_3_day']
    del new_df['consumption_counterpart_all_lag_7_day']
    
    if client_df is not None:
        new_df = new_df.merge(
            full_client.drop("date", axis=1),  
            left_on=["county", "is_business", "product_type", "date"], 
            right_on=["county", "is_business", "product_type", "available_date"], 
            how="left")
    
    new_df = new_df.set_index(['county', 'is_business', 'product_type', 'is_consumption', 'datetime']).sort_index(ascending=True)


    grouped_eic_count = new_df.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['eic_count']
#     new_df['eic_count_2_day_diff'] = grouped_eic_count.shift(0).fillna(0) - grouped_eic_count.shift(24*0).fillna(0)
#     new_df['eic_count_7_day_diff'] = grouped_eic_count.shift(0).fillna(0) - grouped_eic_count.shift(24*5).fillna(0)
#     new_df['eic_count_7_day_mean_diff'] = grouped_eic_count.shift(0).fillna(0) - np.mean([grouped_eic_count.shift(24*i).fillna(0).values for i in range(0, 7)], axis=0)

#     new_df['eic_count_2_day_diff'] = grouped_eic_count.shift(0) - grouped_eic_count.shift(24*0)
    new_df['eic_count_7_day_diff'] = grouped_eic_count.shift(0) - grouped_eic_count.shift(24*5)
    new_df['eic_count_7_day_mean_diff'] = grouped_eic_count.shift(0) - np.mean([grouped_eic_count.shift(24*i).values for i in range(0, 7)], axis=0)

    
    grouped_installed_capacity = new_df.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['installed_capacity']
#     new_df['installed_capacity_2_day_diff'] = grouped_installed_capacity.shift(0).fillna(0) - grouped_installed_capacity.shift(24*0).fillna(0)
#     new_df['installed_capacity_7_day_diff'] = grouped_installed_capacity.shift(0).fillna(0) - grouped_installed_capacity.shift(24*5).fillna(0)
#     new_df['installed_capacity_7_day_mean_diff'] = grouped_installed_capacity.shift(0).fillna(0) - np.mean([grouped_installed_capacity.shift(24*i).fillna(0) for i in range(0, 7)], axis=0)

#     new_df['installed_capacity_2_day_diff'] = grouped_installed_capacity.shift(0) - grouped_installed_capacity.shift(24*0)
    new_df['installed_capacity_7_day_diff'] = grouped_installed_capacity.shift(0) - grouped_installed_capacity.shift(24*5)
    new_df['installed_capacity_7_day_mean_diff'] = grouped_installed_capacity.shift(0) - np.mean([grouped_installed_capacity.shift(24*i) for i in range(0, 7)], axis=0)

    
    
    grouped_target = new_df.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target']
    
    for i in range(2, 22):
        new_df[f'lag_{i}_day'] = grouped_target.shift(24*i).fillna(0)

    
    for i in range(2, 9):
        new_df[f'lag_{i}_day_1h'] = grouped_target.shift(24*i+1).fillna(0)
        
    new_df['lag_7_day_mean'] = np.mean([new_df[f'lag_{i}_day'] for i in range(2, 9)], axis=0) 
    new_df['lag_7_day_1h_mean'] = np.mean([new_df[f'lag_{i}_day_1h'] for i in range(2, 9)], axis=0) 
    new_df['lag_8_14_day_mean'] = np.mean([new_df[f'lag_{i}_day'] for i in range(9, 16)], axis=0)


    new_df['lag_2_day_1h'] = grouped_target.shift(48+1)
    new_df['lag_3_day_1h'] = grouped_target.shift(72+1)
#     new_df['lag_4_day_1h'] = grouped_target.shift(96+1)
#     new_df['lag_7_day_1h'] = grouped_target.shift(24*7+1)
    new_df['lag_7_day_m1h'] = grouped_target.shift(24*7-1)


    grouped_target = new_df.reset_index().groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target']
    
    new_df['rolling_mean_24'] = grouped_target.rolling(24).mean().shift(base_lag).values
    new_df['rolling_std_24'] = grouped_target.rolling(24).std().shift(base_lag).values
    new_df['rolling_kurt_24'] = grouped_target.rolling(24).kurt().shift(base_lag).values
    new_df['rolling_skew_24'] = grouped_target.rolling(24).skew().shift(base_lag).values

    new_df['rolling_mean_24_lag_7_day'] = grouped_target.rolling(24).mean().shift(24*7).values
#     new_df['rolling_mean_24_lag_7_day'] = grouped_target.rolling(24).mean().shift(24*7).values
    new_df['rolling_mean_24_lag_9_day'] = grouped_target.rolling(24).mean().shift(24*9).values
    new_df['ratio_rolling_mean_lag_9_7_day'] = (new_df['rolling_mean_24_lag_9_day'] + 10) /(new_df['rolling_mean_24_lag_7_day'] + 10)

    
    grouped_consumption_counterpart = new_df.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['consumption_counterpart']
    for i in range(2,15):
#     for i in [2, 3, 7, 14]:
        new_df[f'lag_{i}_day_consumption_counterpart'] = grouped_consumption_counterpart.shift(24*i)

    new_df['lag_7_day_mean_consumption_counterpart'] = np.mean([new_df[f'lag_{i}_day_consumption_counterpart'] for i in range(2,9)], axis=0)
    
    
    new_df['ratio_lag_9_2_day'] = (new_df['lag_9_day'] + 10) /(new_df['lag_2_day'] + 10) 
    new_df['ratio_lag_14_7_day'] = (new_df['lag_14_day'] + 10) /(new_df['lag_7_day'] + 10)
    new_df['ratio_lag_9_7_day'] = (new_df['lag_9_day'] + 10) /(new_df['lag_7_day'] + 10)
    new_df['ratio_lag_21_7_day'] = (new_df['lag_21_day'] + 10) /(new_df['lag_7_day'] + 10)
    
    new_df = new_df.reset_index()


    grouped_target_all = new_df[['datetime', 'is_business', 'is_consumption', 'target']].groupby(['is_business', 'is_consumption','datetime'])[['target']].sum()
    grouped_target_all['target_all_lag_2_day'] = grouped_target_all['target'].shift(24*2)
    grouped_target_all['target_all_lag_7_day'] = grouped_target_all['target'].shift(24*7)
    del grouped_target_all['target']
    
    grouped_consumption_counterpart_all = new_df[['datetime', 'county', 'is_business', 'is_consumption', 'consumption_counterpart']].groupby(['is_business', 'is_consumption','datetime'])[['consumption_counterpart']].sum()
    grouped_consumption_counterpart_all['consumption_counterpart_all_lag_2_day'] = grouped_consumption_counterpart_all['consumption_counterpart'].shift(24*2)
    grouped_consumption_counterpart_all['consumption_counterpart_all_lag_3_day'] = grouped_consumption_counterpart_all['consumption_counterpart'].shift(24*3)
    grouped_consumption_counterpart_all['consumption_counterpart_all_lag_7_day'] = grouped_consumption_counterpart_all['consumption_counterpart'].shift(24*7)
    del grouped_consumption_counterpart_all['consumption_counterpart']


    new_df = new_df.merge(
            grouped_target_all,
            on=["datetime", "is_business", 'is_consumption'], 
            how="left")
    
    new_df = new_df.merge(
            grouped_consumption_counterpart_all,
            on=["datetime", "is_business", 'is_consumption'], 
            how="left")

    
    new_df['county'] = new_df['county'].astype(dtype)
    new_df['product_type'] = new_df['product_type'].astype(dtype)
    new_df['dayofweek'] = new_df['datetime'].dt.dayofweek.astype(dtype)
    
    
#     if truncate:
    truncated_new_df = new_df[new_df['datetime'] >= predicting_start_time]
    truncated_new_df = truncated_new_df[truncated_new_df['row_id'].notna()]
    truncated_new_df['row_id'] = truncated_new_df['row_id'].astype(int)

    return truncated_new_df, new_df

In [54]:
import sys
if platform.release() == '5.15.133.1-microsoft-standard-WSL2':
    sys.path.append('./predict-energy-behavior-of-prosumers') 

In [55]:
import enefit
env = enefit.make_env()

## Check the datetime of the test dataframe

In [56]:
def catogorical_to_int_type(df):
    df['is_business'] = df['is_business'].astype(int)
    df['is_consumption'] = df['is_consumption'].astype(int)
    df['county'] = df['county'].astype(int)
    df['product_type'] = df['product_type'].astype(int)
    
    return df

In [57]:
training_cutoff_date = datetime.date(2023,5,31)

In [58]:
history_cutoff = "2023-04-28 00:00:00"
new_data = None
days_with_new_data = 0

online_training = False
debug = False

if debug:
    enefit.make_env.__called__ = False
    type(env)._state = type(type(env)._state).__dict__['INIT']
    
    
iter_test = env.iter_test()

sample_prediction_last = None
sample_prediction_last_last = None

init = False

for (test, revealed_targets, client, historical_weather,
        forecast_weather, electricity_prices, gas_prices, sample_prediction) in iter_test:
    
    currently_scored = test['currently_scored'].sum() > 0
    
    tick = time.time()
#     assert False
    if sample_prediction_last_last is not None:
        print("MAE: ", abs(revealed_targets['target'] - sample_prediction_last_last['target']).mean())
    
    
    test=test.rename(columns={'prediction_datetime':'datetime'})
    
    predicting_start_time = test['datetime'].min()
    
    test['date'] = pd.to_datetime(test.datetime).dt.date
    revealed_targets['date'] = pd.to_datetime(revealed_targets.datetime).dt.date
    predicting_date = predicting_start_time.date()
    
    print(f"Predicting start time: {predicting_start_time}")
    
    
#     placeholder_previous_day = revealed_targets.copy()
#     placeholder_previous_day['datetime'] += datetime.timedelta(days=1)
#     placeholder_previous_day['date'] = placeholder_previous_day['datetime'].dt.date
#     placeholder_previous_day['target'] = np.nan
#     assert False
#     test = pd.concat([test, placeholder_previous_day])
    
    test = catogorical_to_int_type(test)
    revealed_targets = catogorical_to_int_type(revealed_targets)



    # Electricity Prices Data
    # TODO: check data
    if not init:
        cutoff_time = electricity_prices['forecast_date'].min()
        print(electricity_prices['forecast_date'].min())
        full_elec_prices = elec_prices[
            (pd.to_datetime(elec_prices['forecast_date']) >= pd.to_datetime(history_cutoff)) 
            * (pd.to_datetime(elec_prices['forecast_date']) < cutoff_time)
        ]
    full_elec_prices = pd.concat([full_elec_prices, electricity_prices])
    test = test.merge(
        process_elec_prices(full_elec_prices)[["euros_per_mwh", 
                                               'euros_per_mwh_lag_7_day_mean_diff', 
                                               "daily_euros_per_mwh", 
                                                "euros_per_mwh_lag_7_day_diff",
                                                "euros_per_mwh_lag_2_day_diff",
                                               "data_available_datetime",]],
        how="left",
        right_on="data_available_datetime",
        left_on="datetime",
       )
    
    
    revealed_targets = revealed_targets.merge(
        process_elec_prices(full_elec_prices)[["euros_per_mwh", 
                                               'euros_per_mwh_lag_7_day_mean_diff', 
                                               "daily_euros_per_mwh", 
                                                "euros_per_mwh_lag_7_day_diff",
                                                "euros_per_mwh_lag_2_day_diff",
                                               "data_available_datetime",]],
        how="left",
        right_on="data_available_datetime",
        left_on="datetime",
       )
    
    
    # Client Data
    client['date'] = pd.to_datetime(client.date).dt.date
    client['available_date'] = client['date'] + datetime.timedelta(days=2)
    client['is_business'] = client['is_business'].astype(int)
    client['product_type'] = client['product_type'].astype(int)
    client['county'] = client['county'].astype(int)
    
    if not init:
        cutoff_time = client['available_date'].min()
        full_client = client_train[
            (pd.to_datetime(client_train['available_date']) >= pd.to_datetime(history_cutoff))
            * (pd.to_datetime(client_train['available_date']) < pd.to_datetime(cutoff_time))
        ]
    full_client = pd.concat([full_client, client])

#     test = test.merge(
#         full_client.drop("date", axis=1),  
#         left_on=["county", "is_business", "product_type", "date"], 
#         right_on=["county", "is_business", "product_type", "available_date"], 
#         how="left")
    
    historical_weather['datetime'] = pd.to_datetime(historical_weather['datetime'])
    if not init:
        cutoff_time = historical_weather['datetime'].min()
        print(historical_weather['datetime'].min())
        full_historical_weather = historical_weather_train[
            (pd.to_datetime(historical_weather_train['datetime']) >= pd.to_datetime(history_cutoff)) 
            * (pd.to_datetime(historical_weather_train['datetime']) < cutoff_time)
        ]
    full_historical_weather = pd.concat([full_historical_weather, historical_weather])
    
    forecast_weather['forecast_datetime'] = pd.to_datetime(forecast_weather['forecast_datetime'])
    # forecast_weather = correct_weather_bias(forecast_weather)

    
    if not init:
        cutoff_time = forecast_weather['forecast_datetime'].min()
        print(forecast_weather['forecast_datetime'].min())
        full_forecast_weather = forecast_weather_train[
            (forecast_weather_train['forecast_datetime'] >= pd.to_datetime(history_cutoff)) 
            * (forecast_weather_train['forecast_datetime']< cutoff_time)
        ]
    full_forecast_weather = pd.concat([full_forecast_weather, forecast_weather])
    
    
    ttick = time.time()
    test = test.merge(
        get_weighted_forecast_by_county(full_forecast_weather,
                                        get_weighted_historical_weather_by_county(full_historical_weather),
                                        filter_hours_ahead=False),
        how="left",
        on=["county", "datetime",],
    )
    
    revealed_targets = revealed_targets.merge(
        get_weighted_forecast_by_county(full_forecast_weather,  
                                        get_weighted_historical_weather_by_county(full_historical_weather),
                                       filter_hours_ahead=False),
        how="left",
        on=["county", "datetime",],
    )
    
    print(time.time() - ttick,"sssss")

    test["month"]  = test["datetime"].dt.month
    test['day'] = test['datetime'].dt.day
    test['dayofweek'] = test['datetime'].dt.dayofweek
    
    test['hour'] = test['datetime'].dt.hour
    test['sin_hour'] = np.sin(2*np.pi*test.hour/hours_in_day)
    test['cos_hour'] = np.cos(2*np.pi*test.hour/hours_in_day)
    test['hour'] = test['hour'].astype(dtype)
    
    test['sin_month'] = np.sin(2*np.pi*test.month/months_in_year)
    test['cos_month'] = np.cos(2*np.pi*test.month/months_in_year)
    test["month"]  = test["month"].astype(dtype)
    
    revealed_targets["month"]  = revealed_targets["datetime"].dt.month
    revealed_targets['dayofweek'] = revealed_targets['datetime'].dt.dayofweek

    revealed_targets['hour'] = revealed_targets['datetime'].dt.hour
    revealed_targets['sin_hour'] = np.sin(2*np.pi*revealed_targets.hour/hours_in_day)
    revealed_targets['cos_hour'] = np.cos(2*np.pi*revealed_targets.hour/hours_in_day)
    revealed_targets['hour'] = revealed_targets['hour'].astype(dtype)
    
    revealed_targets['sin_month'] = np.sin(2*np.pi*revealed_targets.month/months_in_year)
    revealed_targets['cos_month'] = np.cos(2*np.pi*revealed_targets.month/months_in_year)


    if not init:
        cutoff_time = revealed_targets['datetime'].min()
        print(revealed_targets['datetime'].min())
        historic_train = train[
            (train['datetime'] >= pd.to_datetime(history_cutoff)) * (train['datetime'] < cutoff_time)
        ]
    historic_train = pd.concat([historic_train, revealed_targets])
    
    placeholder_previous_day = revealed_targets.copy()
    placeholder_previous_day['datetime'] += datetime.timedelta(days=1)
    placeholder_previous_day['target'] = np.nan
    
    
    test['target'] = np.nan
#     assert False
    
    test, untruncated_test = append_lag(test, placeholder_previous_day, historic_train, predicting_start_time=predicting_start_time, client_df=full_client)
    
                
                
    for col in test.columns:
        if col in train.columns:
            if not test[col].dtype == train[col].dtype:
#                 print(col, test[col].dtype, train[col].dtype )
                test[col] = test[col].astype(train[col].dtype)

            if test[col].dtype in [np.float32, np.float64, float]:
    #             print('clipping')
                test[col].clip(-1000000,1000000)
    
    if online_training and currently_scored and (predicting_date > training_cutoff_date): # !!!TODO

        untruncated_test = append_diffs(untruncated_test)
        
        
        new_data = pd.concat([
            new_data,
            untruncated_test[
                            (untruncated_test['datetime'].dt.date == revealed_targets['datetime'].dt.date[0])
                            * (~untruncated_test['target'].isna())
                        ]
        ])
        
#         assert False
        days_with_new_data += 1
        
        if days_with_new_data >= 14:
            
#             new_data_augmented = scale_features(new_data, growth_factor=0.1, n_rows=len(new_data))
#             new_data_augmented = pd.concat([new_data, new_data_augmented])

            
            continue_training(
                results,
#                 new_data = new_data_augmented
                new_data = new_data
            )
#             assert False
            # clear new data
            new_data = None
            days_with_new_data = 0

    

    test = test.sort_values('row_id', ascending=True)
    
    
    tock = time.time()
    if not currently_scored:
        sample_prediction['target'] = 0.0
    else:
    
        preds = get_prediciton(test, results, scaled=True, epsilon=100)
        #     preds_2 = get_prediciton(test, results_2, scaled=True, epsilon=10)
        #     preds[-1]['pred'] = preds[-1]['pred']*0.5 + preds_2[-1]['pred']*0.5

        test = preds[-1]
        sample_prediction['target'] = test['pred'].reset_index(drop=True).fillna(0).values.astype(float)

    
    sample_prediction_last_last = sample_prediction_last
    sample_prediction_last = sample_prediction
    
    
    env.predict(sample_prediction)
    print(f"predicted {len(sample_prediction)} rows in {time.time() - tick:.1f}s, inferercen time: {time.time() - tock:.1f}s")
    init = True
    
#     assert False


This version of the API is not optimized and should not be used to estimate the runtime of your code on the hidden test set.
Predicting start time: 2023-05-28 00:00:00
2023-05-27 00:00:00
2023-05-26 11:00:00
2023-05-27 03:00:00
2.428032159805298 sssss
2023-05-26 00:00:00
predicted 3120 rows in 6.8s, inferercen time: 0.0s
Predicting start time: 2023-05-29 00:00:00
2.402756690979004 sssss
predicted 3120 rows in 6.2s, inferercen time: 0.0s
MAE:  378.32391538461536
Predicting start time: 2023-05-30 00:00:00
2.521319627761841 sssss
predicted 3120 rows in 6.3s, inferercen time: 0.0s
MAE:  392.32576858974363
Predicting start time: 2023-05-31 00:00:00
2.601292848587036 sssss
predicted 3120 rows in 6.5s, inferercen time: 0.0s


In [59]:
# for key in keys:
#     for is_consumption in [0, 1]:
#         boosters = get_boosters(key)

#         for ft in boosters[0].feature_name():
#             if ("eu" in ft):
#                 print(key)
#                 print(ft)

In [60]:
# for col in test.columns:
#     if col in train.columns:
#         if not test[col].dtype == train[col].dtype:
#             print(col, test[col].dtype, train[col].dtype )


In [61]:
# %%time

# for col in test.columns:
#     if col in train.columns:
#         if not test[col].dtype == train[col].dtype:
#             print(col, test[col].dtype, train[col].dtype )
#             test[col] = test[col].astype(train[col].dtype)
            
#         if test[col].dtype in [np.float32, np.float64, float]:
# #             print('clipping')
#             test[col].clip(-1000000,1000000)

In [62]:
train.dtypes[train.dtypes == 'category']

county                    category
product_type              category
month                     category
dayofweek                 category
daytype                   category
day_category              category
daytype_lag_2_day         category
daytype_lag_3_day         category
daytype_lag_7_day         category
day_category_lag_2_day    category
day_category_lag_3_day    category
day_category_lag_7_day    category
hour                      category
dtype: object

In [63]:
test.dtypes[test.dtypes == 'category']

county                    category
product_type              category
month                     category
dayofweek                 category
hour                      category
daytype                   category
day_category              category
daytype_lag_2_day         category
daytype_lag_3_day         category
daytype_lag_7_day         category
day_category_lag_2_day    category
day_category_lag_3_day    category
day_category_lag_7_day    category
dtype: object

In [64]:
# for key in results.keys():
#     print(key, results[key]['cvbooster'].boosters[0].num_trees())


In [65]:
# for col in untruncated_test.columns:
#     if untruncated_test[col].dtype == np.float16:
#         print(col)

In [66]:
test.isna().sum(axis=0)[test.isna().sum(axis=0) > 0]

target                                  3120
data_block_id_x                         3120
consumption_counterpart                 3120
lag_2_day_1h_consumption_counterpart    3120
ratio_lag_10_3_day                      3120
yearmonth                               3120
yearmonthday                            3120
lag_7_day_mean_diff                     3120
lag_7_day_mean_diff_neg                 3120
lag_7_day_1h_mean_diff                  3120
lag_7_day_1h_mean_diff_neg              3120
lag_2_day_diff                          3120
lag_2_day_diff_neg                      3120
lag_3_day_diff                          3120
lag_3_day_diff_neg                      3120
lag_7_day_diff                          3120
lag_7_day_diff_neg                      3120
lag_14_day_diff                         3120
lag_14_day_diff_neg                     3120
scaled_target                           3120
scaled_lag_7_day_mean_diff              3120
scaled_lag_7_day_mean_diff_neg          3120
scaled_lag

In [67]:
test.isna().sum(axis=0)[test.isna().sum(axis=0) == 0]

county                                   0
is_business                              0
product_type                             0
is_consumption                           0
datetime                                 0
                                        ..
target_all_lag_2_day                     0
target_all_lag_7_day                     0
consumption_counterpart_all_lag_2_day    0
consumption_counterpart_all_lag_3_day    0
consumption_counterpart_all_lag_7_day    0
Length: 241, dtype: int64

In [68]:
all_features = []

for key, boosters in results.items():
#     all_features += boosters.feature_name()
    all_features += boosters['cvbooster'].boosters[0].feature_name()
    
all_features = list(set(all_features))

In [69]:
test[all_features].describe().iloc[:, :50].T

,count,mean,std,min,25%,50%,75%,max
rolling_skew_24,3120.0,5.403876e-01,0.490111,-1.446347,0.289057,6.077764e-01,7.885602e-01,3.052810
surface_solar_radiation_downwards_7_day_mean_diff,3120.0,1.559973e+01,73.672638,-285.445496,0.000000,7.691986e+00,3.617765e+01,205.333633
cos_hour,3120.0,-7.287618e-17,0.707220,-1.000000,-0.707107,-6.123234e-17,7.071068e-01,1.000000
cloudcover_total_3_day_diff,3120.0,-1.245499e+01,15.277463,-72.614716,-20.968828,-5.743967e+00,-3.484493e-01,0.584365
eic_count_7_day_diff,3120.0,2.307692e-01,1.186902,-1.000000,0.000000,0.000000e+00,0.000000e+00,9.000000
ratio_lag_14_7_day,3120.0,9.819696e-01,0.472404,0.056828,0.748079,9.969708e-01,1.143940e+00,4.914949
sin_month,3120.0,5.000000e-01,0.000000,0.500000,0.500000,5.000000e-01,5.000000e-01,0.500000
diffuse_radiation_lag_7_day_h,3120.0,6.920415e+01,59.563931,0.000000,0.040235,6.875746e+01,1.207745e+02,200.706406
temperature,3120.0,1.286724e+01,4.136602,5.851340,8.606516,1.343425e+01,1.649855e+01,19.542820
lag_4_day,3120.0,3.842370e+02,1039.807373,0.000000,11.021250,6.356400e+01,2.842750e+02,11013.487305


In [70]:
test[all_features].describe().iloc[:, 50:100].T

,count,mean,std,min,25%,50%,75%,max
surface_solar_radiation_downwards_3_day_diff,3120.0,-1.908322e+01,78.352997,-429.965668,-4.733311,0.000000e+00,3.635755,273.251160
sin_hour,3120.0,0.000000e+00,0.707220,-1.000000,-0.707107,6.123234e-17,0.707107,1.000000
consumption_counterpart_all_lag_3_day,3120.0,1.277359e+04,14133.658203,44.166000,1332.834961,7.377790e+03,19150.943359,52905.339844
lag_7_day_1h_mean,3120.0,3.837182e+02,997.661011,0.000000,11.571464,7.221814e+01,339.750114,8449.158203
lag_7_day_m1h,3120.0,3.997113e+02,1055.875732,0.000000,8.908750,6.898850e+01,346.954002,9111.477539
cloudcover_total,3120.0,3.107824e-01,0.264176,0.000000,0.094323,2.383707e-01,0.462732,0.985331
diffuse_radiation_lag_3_day_h,3120.0,6.500838e+01,52.842850,0.000000,0.556049,7.573769e+01,111.749249,252.978973
installed_capacity_7_day_diff,3120.0,8.779988e+00,43.530319,-14.500430,-0.000024,0.000000e+00,0.000015,320.000000
cloudcover_low,3120.0,4.141657e-02,0.108163,0.000000,0.000000,0.000000e+00,0.012727,0.734994
temperature_8_14_day_mean_diff,3120.0,2.936178e-02,1.420875,-3.288605,-1.183676,3.093543e-01,1.073747,3.385558


In [71]:
test[all_features].describe().iloc[:, 100:].T

,count,mean,std,min,25%,50%,75%,max


In [72]:
# for col in sorted(test.columns):
#     if not col in all_features:
#         print(col)